# Day 14 — Chronological Benchmark Construction

Freeze the temporal benchmark and historical-only training and evaluation protocols.

In [ ]:
# Imports
import json
from pathlib import Path

In [ ]:
# Project paths
def find_project_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "README.md").exists():
            return path
    raise RuntimeError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

def resolve_artifact_paths(value):
    """Rebase legacy project paths in loaded metadata; never rewrite artifacts.

    Keep all non-path values, model identifiers, and scientific settings intact.
    Support the original project name and the public repository name.
    """
    if isinstance(value, dict):
        return {key: resolve_artifact_paths(item) for key, item in value.items()}
    if isinstance(value, list):
        return [resolve_artifact_paths(item) for item in value]
    if isinstance(value, str):
        normalized = value.replace("\\", "/")
        for name in ("FinGPT_Project", "FinGPT-Temporal-Generalization"):
            marker = "/" + name + "/"
            if marker in normalized and "\n" not in normalized:
                # Only standalone filesystem references, never prose containing a path.
                prefix, relative = normalized.split(marker, 1)
                if prefix.startswith("/") or (len(prefix) > 1 and prefix[1] == ":"):
                    if not any(char.isspace() for char in relative):
                        return str(PROJECT_ROOT / relative)
    return value

def load_artifact_json(stream, *args, **kwargs):
    return resolve_artifact_paths(json.load(stream, *args, **kwargs))


In [ ]:
# Reproducibility settings
# Original seeds and deterministic settings remain at their original workflow
# positions below, including stage-specific RNG resets. No new seed is introduced.

**Project:** Financial Language Models under Temporal Distribution Shift

## Objective

Pre-specify and freeze a defensible **within-Dataset-B chronological benchmark** before any new LoRA training.

Day 14 will:

- lock the 39,400-observation dense Dataset B benchmark population;
- compare candidate historical cutoffs using only pre-model information;
- freeze chronological train / validation / future-window boundaries;
- freeze the Dataset-B LoRA training protocol;
- freeze evaluation metrics and temporal-degradation analysis rules;
- rerun the leakage guard at the final selected boundary.

**Day 14 does not train or evaluate a new model.**

Target workflow:

Historical Train → Historical Validation → Freeze Model → Future F1 → F2 → ... → FK

In [ ]:
# Load and validate required Day 07 / Day 13 inputs
from pathlib import Path
import json
import pandas as pd

RESULTS_ROOT = PROJECT_ROOT / "results"
DAY07_DIR = RESULTS_ROOT / "day07"
DAY13_DIR = RESULTS_ROOT / "day13"
DAY14_DIR = RESULTS_ROOT / "day14"

DAY14_DIR.mkdir(parents=True, exist_ok=True)


INPUT_PATHS = {
    "temporal_dataset_clean":
        DAY07_DIR / "temporal_dataset_clean.csv",

    "dataset_source_audit":
        DAY13_DIR / "dataset_source_audit.json",

    "temporal_schema_audit":
        DAY13_DIR / "temporal_schema_audit.csv",

    "label_provenance_audit":
        DAY13_DIR / "label_provenance_audit.json",

    "timestamp_audit":
        DAY13_DIR / "timestamp_audit.json",

    "temporal_sample_counts":
        DAY13_DIR / "temporal_sample_counts.csv",

    "temporal_label_distribution":
        DAY13_DIR / "temporal_label_distribution.csv",

    "duplicate_leakage_audit":
        DAY13_DIR / "duplicate_leakage_audit.json",

    "benchmark_validity_summary":
        DAY13_DIR / "benchmark_validity_summary.json",
}

print("=" * 80)
print("DAY 14 — UPSTREAM INPUT CHECKPOINT")
print("=" * 80)

missing = []

for name, path in INPUT_PATHS.items():
    if path.exists():
        print(f"PASS — {name}")
    else:
        print(f"FAIL — {name}")
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "\nMissing required Day 14 inputs:\n"
        + "\n".join(f"- {name}" for name in missing)
    )


temporal_df = pd.read_csv(
    INPUT_PATHS["temporal_dataset_clean"]
)


temporal_schema_audit = pd.read_csv(
    INPUT_PATHS["temporal_schema_audit"]
)

day13_sample_counts = pd.read_csv(
    INPUT_PATHS["temporal_sample_counts"]
)

day13_label_distribution = pd.read_csv(
    INPUT_PATHS["temporal_label_distribution"]
)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return load_artifact_json(f)

dataset_source_audit = load_json(
    INPUT_PATHS["dataset_source_audit"]
)

label_provenance_audit = load_json(
    INPUT_PATHS["label_provenance_audit"]
)

timestamp_audit = load_json(
    INPUT_PATHS["timestamp_audit"]
)

duplicate_leakage_audit = load_json(
    INPUT_PATHS["duplicate_leakage_audit"]
)

benchmark_validity_summary = load_json(
    INPUT_PATHS["benchmark_validity_summary"]
)
# 6. Basic load checkpoint
print("\n" + "-" * 80)
print("Loaded canonical Dataset B:")
print(f"Rows    : {len(temporal_df):,}")
print(f"Columns : {temporal_df.shape[1]}")

print("\nDataset columns:")
print(list(temporal_df.columns))

print("\nDay 13 audit artifacts loaded: PASS")
print(f"Day 14 output directory       : {DAY14_DIR}")

print("=" * 80)

## Stage A — Lock the Benchmark Population

The raw Dataset B file contains 39,965 observations. Day 13 established that the primary temporal benchmark should use only the dense regime:

- start: 2025-06-03 21:06:16 UTC
- end: 2026-06-05 12:59:13 UTC
- expected N: 39,400

The 565 earlier observations are excluded from the core benchmark because they belong to a sparse and substantially different source regime.

This cell constructs the canonical Day 14 benchmark population. All subsequent cutoff selection, chronological splitting, source diagnostics, and future-window definitions must operate on this locked population only.

In [ ]:
# construct and validate canonical benchmark population
# 1. Parse timestamps
temporal_df["date"] = pd.to_datetime(
    temporal_df["date"],
    utc=True,
    errors="coerce"
)

if temporal_df["date"].isna().any():
    raise ValueError(
        f"Invalid timestamps detected: "
        f"{temporal_df['date'].isna().sum():,}"
    )
# 2. Freeze dense-regime boundaries from Day 13
DENSE_START = pd.Timestamp(
    "2025-06-03 21:06:16",
    tz="UTC"
)

DENSE_END = pd.Timestamp(
    "2026-06-05 12:59:13",
    tz="UTC"
)

EXPECTED_DENSE_N = 39_400
EXPECTED_EXCLUDED_N = 565
# 3. Construct canonical benchmark population
benchmark_df = (
    temporal_df.loc[
        temporal_df["date"].between(
            DENSE_START,
            DENSE_END,
            inclusive="both"
        )
    ]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

excluded_df = temporal_df.loc[
    ~temporal_df.index.isin(
        temporal_df.loc[
            temporal_df["date"].between(
                DENSE_START,
                DENSE_END,
                inclusive="both"
            )
        ].index
    )
].copy()
# 4. Core validation checks
checks = {
    "dense_population_n":
        len(benchmark_df) == EXPECTED_DENSE_N,

    "excluded_population_n":
        len(excluded_df) == EXPECTED_EXCLUDED_N,

    "chronological_order":
        benchmark_df["date"].is_monotonic_increasing,

    "exact_start_timestamp":
        benchmark_df["date"].min() == DENSE_START,

    "exact_end_timestamp":
        benchmark_df["date"].max() == DENSE_END,

    "three_sentiment_classes":
        set(benchmark_df["sentiment"].dropna().unique())
        == {"negative", "neutral", "positive"},

    "sentiment_complete":
        benchmark_df["sentiment"].notna().all(),

    "channel_available":
        "channel_username" in benchmark_df.columns,

    "text_complete":
        benchmark_df["text"].notna().all(),
}
# 5. Report benchmark population
print("=" * 80)
print("DAY 14 — STAGE A: BENCHMARK POPULATION")
print("=" * 80)

print(f"\nFull Dataset B N      : {len(temporal_df):,}")
print(f"Dense benchmark N     : {len(benchmark_df):,}")
print(f"Excluded early N      : {len(excluded_df):,}")

print(
    f"\nBenchmark start       : "
    f"{benchmark_df['date'].min()}"
)

print(
    f"Benchmark end         : "
    f"{benchmark_df['date'].max()}"
)

print("\nSentiment counts:")
print(
    benchmark_df["sentiment"]
    .value_counts()
    .reindex(["negative", "neutral", "positive"])
    .to_string()
)

print(
    f"\nUnique channels       : "
    f"{benchmark_df['channel_username'].nunique(dropna=True):,}"
)

print("\nValidation checks:")
for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} — {name}")
# 6. Hard checkpoint
if not all(checks.values()):
    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nStage A benchmark-population validation failed:\n"
        + "\n".join(f"- {name}" for name in failed)
    )

print("\n" + "-" * 80)
print("STAGE A CORE POPULATION CHECK: PASS")
print(
    "Canonical Day 14 benchmark population "
    "is locked at 39,400 observations."
)
print("=" * 80)

### Stage A Output — Benchmark Population Manifest

The Day 14 benchmark population has passed all structural checks and is now frozen.

The manifest records:

- canonical population size;
- exact UTC date range;
- class counts and proportions;
- source/channel counts and proportions;
- the exclusion rule for the sparse pre-June-2025 observations;
- provenance linking the benchmark back to the Day 07 cleaned Dataset B and Day 13 validity audit.

All subsequent Day 14 stages must use this population without modification.

In [ ]:
# Stage A output: benchmark population manifest
# 1. Class distribution
class_counts = (
    benchmark_df["sentiment"]
    .value_counts()
    .reindex(["negative", "neutral", "positive"])
)

class_distribution = {
    label: {
        "count": int(class_counts[label]),
        "proportion": float(class_counts[label] / len(benchmark_df))
    }
    for label in class_counts.index
}

# 2. Source/channel distribution
source_counts = (
    benchmark_df["channel_username"]
    .fillna("<MISSING>")
    .value_counts()
)

source_distribution = {
    str(channel): {
        "count": int(count),
        "proportion": float(count / len(benchmark_df))
    }
    for channel, count in source_counts.items()
}
# 3. Build manifest
benchmark_population_manifest = {
    "day": 14,
    "artifact": "benchmark_population_manifest",
    "project": "Financial Language Models under Temporal Distribution Shift",

    "benchmark_population": {
        "dataset": "Dataset B",
        "population_rule": "Day 13 dense temporal regime only",
        "n_observations": int(len(benchmark_df)),
        "start_timestamp_utc": benchmark_df["date"].min().isoformat(),
        "end_timestamp_utc": benchmark_df["date"].max().isoformat(),
        "chronologically_sorted": bool(
            benchmark_df["date"].is_monotonic_increasing
        )
    },

    "excluded_population": {
        "n_observations": int(len(excluded_df)),
        "reason": (
            "Sparse pre-June-2025 observations excluded from the "
            "primary benchmark because Day 13 identified them as "
            "belonging to a substantially different and sparse "
            "source regime."
        )
    },

    "label_schema": {
        "target_column": "sentiment",
        "classes": ["negative", "neutral", "positive"],
        "label_provenance": "LLM-generated sentiment labels; not human gold",
        "distribution": class_distribution
    },

    "source_schema": {
        "source_column": "channel_username",
        "n_unique_sources": int(
            benchmark_df["channel_username"].nunique(dropna=True)
        ),
        "distribution": source_distribution
    },

    "required_columns": [
        "message_id",
        "channel_username",
        "date",
        "text",
        "sentiment"
    ],

    "upstream_provenance": {
        "dataset_source": str(
            INPUT_PATHS["temporal_dataset_clean"]
        ),
        "day13_dataset_source_audit": str(
            INPUT_PATHS["dataset_source_audit"]
        ),
        "day13_label_provenance_audit": str(
            INPUT_PATHS["label_provenance_audit"]
        ),
        "day13_timestamp_audit": str(
            INPUT_PATHS["timestamp_audit"]
        ),
        "day13_duplicate_leakage_audit": str(
            INPUT_PATHS["duplicate_leakage_audit"]
        ),
        "day13_benchmark_validity_summary": str(
            INPUT_PATHS["benchmark_validity_summary"]
        )
    },

    "freeze_status": "LOCKED",
    "downstream_rule": (
        "All Day 14 cutoff, split, future-window, source-composition, "
        "and leakage analyses must operate only on this benchmark population."
    )
}

# 4. Save manifest
manifest_path = DAY14_DIR / "benchmark_population_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        benchmark_population_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

# 5. Artifact checkpoint
print("=" * 80)
print("DAY 14 — STAGE A ARTIFACT")
print("=" * 80)

print(f"\nSaved:\n{manifest_path}")
print(f"\nExists : {manifest_path.exists()}")

if manifest_path.exists():
    print(f"Size   : {manifest_path.stat().st_size:,} bytes")

print("\nFrozen benchmark summary:")
print(f"N                 : {len(benchmark_df):,}")
print(
    f"Date range        : "
    f"{benchmark_df['date'].min()} -> "
    f"{benchmark_df['date'].max()}"
)
print(
    f"Unique channels   : "
    f"{benchmark_df['channel_username'].nunique(dropna=True)}"
)

print("\nClass distribution:")
for label, stats in class_distribution.items():
    print(
        f"{label:<8} "
        f"N={stats['count']:>6,} "
        f"({stats['proportion']:.4f})"
    )

if not manifest_path.exists():
    raise RuntimeError(
        "benchmark_population_manifest.json was not saved."
    )

print("\n" + "-" * 80)
print("STAGE A: PASS — BENCHMARK POPULATION FROZEN")
print("=" * 80)

## Stage B — Candidate Historical Cutoff Feasibility

Candidate historical cutoffs are evaluated using only pre-model information.

The candidate boundaries are:

- 2025-10-01
- 2025-11-01
- 2025-12-01
- 2026-01-01
- 2026-02-01

For each cutoff, evaluate:

- historical sample size;
- future sample size;
- historical and future class support;
- number of strictly future calendar windows;
- source overlap between historical and future periods;
- source-composition shift.

No LoRA predictions, accuracy, F1, or other model-performance metrics are permitted in this stage.

In [ ]:
# Stage B: candidate historical cutoff feasibility
import numpy as np
import pandas as pd

CANDIDATE_CUTOFFS = [
    pd.Timestamp("2025-10-01", tz="UTC"),
    pd.Timestamp("2025-11-01", tz="UTC"),
    pd.Timestamp("2025-12-01", tz="UTC"),
    pd.Timestamp("2026-01-01", tz="UTC"),
    pd.Timestamp("2026-02-01", tz="UTC"),
]

SENTIMENT_CLASSES = ["negative", "neutral", "positive"]

def total_variation_distance(p, q):
    """
    Compute TVD between two categorical distributions.
    """
    labels = sorted(set(p.index).union(set(q.index)))

    p_vec = p.reindex(labels, fill_value=0).astype(float)
    q_vec = q.reindex(labels, fill_value=0).astype(float)

    if p_vec.sum() > 0:
        p_vec = p_vec / p_vec.sum()

    if q_vec.sum() > 0:
        q_vec = q_vec / q_vec.sum()

    return 0.5 * np.abs(p_vec - q_vec).sum()


def count_future_calendar_windows(future_df):
    """
    Count calendar-month windows represented in the strictly future period.
    """
    if future_df.empty:
        return 0

    return (
        future_df["date"]
        .dt.to_period("M")
        .nunique()
    )

rows = []

for cutoff in CANDIDATE_CUTOFFS:

    historical = benchmark_df.loc[
        benchmark_df["date"] < cutoff
    ].copy()

    future = benchmark_df.loc[
        benchmark_df["date"] >= cutoff
    ].copy()

    hist_class_counts = (
        historical["sentiment"]
        .value_counts()
        .reindex(SENTIMENT_CLASSES, fill_value=0)
    )

    future_class_counts = (
        future["sentiment"]
        .value_counts()
        .reindex(SENTIMENT_CLASSES, fill_value=0)
    )

    hist_sources = set(
        historical["channel_username"]
        .dropna()
        .unique()
    )

    future_sources = set(
        future["channel_username"]
        .dropna()
        .unique()
    )

    source_union = hist_sources | future_sources
    source_intersection = hist_sources & future_sources

    source_jaccard = (
        len(source_intersection) / len(source_union)
        if len(source_union) > 0
        else np.nan
    )


    hist_source_dist = (
        historical["channel_username"]
        .fillna("<MISSING>")
        .value_counts()
    )

    future_source_dist = (
        future["channel_username"]
        .fillna("<MISSING>")
        .value_counts()
    )

    source_tvd = total_variation_distance(
        hist_source_dist,
        future_source_dist
    )

    n_future_windows = count_future_calendar_windows(
        future
    )

    rows.append({
        "cutoff_utc": cutoff.isoformat(),

        "historical_n":
            int(len(historical)),

        "future_n":
            int(len(future)),

        "historical_fraction":
            float(len(historical) / len(benchmark_df)),

        "future_fraction":
            float(len(future) / len(benchmark_df)),

        "historical_negative_n":
            int(hist_class_counts["negative"]),

        "historical_neutral_n":
            int(hist_class_counts["neutral"]),

        "historical_positive_n":
            int(hist_class_counts["positive"]),

        "future_negative_n":
            int(future_class_counts["negative"]),

        "future_neutral_n":
            int(future_class_counts["neutral"]),

        "future_positive_n":
            int(future_class_counts["positive"]),

        "historical_min_class_n":
            int(hist_class_counts.min()),

        "future_min_class_n":
            int(future_class_counts.min()),

        "historical_sources":
            int(len(hist_sources)),

        "future_sources":
            int(len(future_sources)),

        "shared_sources":
            int(len(source_intersection)),

        "source_overlap_jaccard":
            float(source_jaccard),

        "source_composition_tvd":
            float(source_tvd),

        "future_calendar_windows":
            int(n_future_windows),
    })

candidate_cutoff_feasibility = pd.DataFrame(rows)

candidate_cutoff_feasibility["all_historical_classes_present"] = (
    candidate_cutoff_feasibility[
        [
            "historical_negative_n",
            "historical_neutral_n",
            "historical_positive_n",
        ]
    ].min(axis=1) > 0
)

candidate_cutoff_feasibility["all_future_classes_present"] = (
    candidate_cutoff_feasibility[
        [
            "future_negative_n",
            "future_neutral_n",
            "future_positive_n",
        ]
    ].min(axis=1) > 0
)

candidate_cutoff_feasibility["basic_feasibility_pass"] = (
    candidate_cutoff_feasibility[
        "all_historical_classes_present"
    ]
    &
    candidate_cutoff_feasibility[
        "all_future_classes_present"
    ]
    &
    (
        candidate_cutoff_feasibility[
            "future_calendar_windows"
        ] >= 4
    )
)

display_columns = [
    "cutoff_utc",
    "historical_n",
    "future_n",
    "historical_min_class_n",
    "future_min_class_n",
    "future_calendar_windows",
    "historical_sources",
    "future_sources",
    "shared_sources",
    "source_overlap_jaccard",
    "source_composition_tvd",
    "basic_feasibility_pass",
]

print("=" * 120)
print("DAY 14 — STAGE B: CANDIDATE CUTOFF FEASIBILITY")
print("=" * 120)

print(
    candidate_cutoff_feasibility[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nCandidate count:",
      len(candidate_cutoff_feasibility))

print(
    "Basic feasibility passes:",
    int(
        candidate_cutoff_feasibility[
            "basic_feasibility_pass"
        ].sum()
    ),
    "/",
    len(candidate_cutoff_feasibility)
)

print("=" * 120)

## Stage C — Freeze the Historical Development Boundary

All five candidate cutoffs satisfy the basic structural feasibility requirements.

The final cutoff is selected using **pre-model information only**.

### Selection rule

A candidate must:

1. contain all three sentiment classes in both historical and future periods;
2. retain at least 15,000 observations in both historical and future periods;
3. retain at least 2,500 observations in the smallest sentiment class on both sides;
4. retain at least 5 strictly future calendar-month windows;
5. retain source-overlap Jaccard ≥ 0.80.

Among candidates satisfying all criteria, select the cutoff that minimizes the absolute imbalance between historical and future sample sizes.

No model predictions, validation scores, accuracy, F1, or Day 08–11 performance are used in this decision.

In [ ]:
# Stage C: freeze historical development boundary
# 1. Pre-declared feasibility thresholds

MIN_HISTORICAL_N = 15_000
MIN_FUTURE_N = 15_000

MIN_CLASS_N = 2_500

MIN_FUTURE_WINDOWS = 5
MIN_SOURCE_OVERLAP = 0.80

# 2. Apply final selection rule

candidate_cutoff_feasibility[
    "final_rule_pass"
] = (
    candidate_cutoff_feasibility[
        "all_historical_classes_present"
    ]
    &
    candidate_cutoff_feasibility[
        "all_future_classes_present"
    ]
    &
    (
        candidate_cutoff_feasibility[
            "historical_n"
        ] >= MIN_HISTORICAL_N
    )
    &
    (
        candidate_cutoff_feasibility[
            "future_n"
        ] >= MIN_FUTURE_N
    )
    &
    (
        candidate_cutoff_feasibility[
            "historical_min_class_n"
        ] >= MIN_CLASS_N
    )
    &
    (
        candidate_cutoff_feasibility[
            "future_min_class_n"
        ] >= MIN_CLASS_N
    )
    &
    (
        candidate_cutoff_feasibility[
            "future_calendar_windows"
        ] >= MIN_FUTURE_WINDOWS
    )
    &
    (
        candidate_cutoff_feasibility[
            "source_overlap_jaccard"
        ] >= MIN_SOURCE_OVERLAP
    )
)

# 3. Quantify historical/future balance
candidate_cutoff_feasibility[
    "sample_size_imbalance"
] = (
    candidate_cutoff_feasibility[
        "historical_n"
    ]
    -
    candidate_cutoff_feasibility[
        "future_n"
    ]
).abs()

eligible_candidates = (
    candidate_cutoff_feasibility.loc[
        candidate_cutoff_feasibility[
            "final_rule_pass"
        ]
    ]
    .copy()
)

if eligible_candidates.empty:
    raise RuntimeError(
        "No candidate cutoff satisfies the frozen Stage C rule."
    )
# 4. Select most balanced eligible cutoff
selected_row = (
    eligible_candidates
    .sort_values(
        [
            "sample_size_imbalance",
            "cutoff_utc"
        ]
    )
    .iloc[0]
)

SELECTED_CUTOFF = pd.Timestamp(
    selected_row["cutoff_utc"]
)
# 5. Save candidate feasibility artifact
candidate_path = (
    DAY14_DIR /
    "candidate_cutoff_feasibility.csv"
)

candidate_cutoff_feasibility.to_csv(
    candidate_path,
    index=False
)
# 6. Report decision
print("=" * 100)
print("DAY 14 — STAGE C: HISTORICAL CUTOFF SELECTION")
print("=" * 100)

report_columns = [
    "cutoff_utc",
    "historical_n",
    "future_n",
    "historical_min_class_n",
    "future_min_class_n",
    "future_calendar_windows",
    "source_overlap_jaccard",
    "sample_size_imbalance",
    "final_rule_pass",
]

print(
    candidate_cutoff_feasibility[
        report_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "-" * 100)

print(
    f"Eligible candidates : "
    f"{len(eligible_candidates)} / "
    f"{len(candidate_cutoff_feasibility)}"
)

print(
    f"Selected cutoff     : "
    f"{SELECTED_CUTOFF}"
)

print(
    f"Historical N        : "
    f"{int(selected_row['historical_n']):,}"
)

print(
    f"Future N            : "
    f"{int(selected_row['future_n']):,}"
)

print(
    f"Sample imbalance    : "
    f"{int(selected_row['sample_size_imbalance']):,}"
)

print(
    f"Future windows      : "
    f"{int(selected_row['future_calendar_windows'])}"
)

print(
    f"Source overlap      : "
    f"{selected_row['source_overlap_jaccard']:.4f}"
)

print(
    f"\nSaved:\n{candidate_path}"
)

EXPECTED_SELECTED_CUTOFF = pd.Timestamp(
    "2026-01-01",
    tz="UTC"
)

if SELECTED_CUTOFF != EXPECTED_SELECTED_CUTOFF:
    raise AssertionError(
        "Selected cutoff differs from the expected "
        "2026-01-01 boundary. Inspect feasibility logic."
    )

if not candidate_path.exists():
    raise RuntimeError(
        "candidate_cutoff_feasibility.csv was not saved."
    )

print("\n" + "-" * 100)
print(
    "STAGE C: PASS — HISTORICAL CUTOFF FROZEN "
    "AT 2026-01-01 UTC"
)
print("=" * 100)

## Stage D — Historical Train / Validation Split

The historical development period is now frozen as:

**2025-06-03 21:06:16 UTC → 2025-12-31 23:59:59 UTC**

Within this historical block, training and validation must also follow chronology:

Historical Train → Historical Validation → Future Evaluation

Candidate validation starts are compared using only:

- train and validation sample size;
- class support;
- source support and overlap;
- number of validation calendar months.

No model performance is used to select the train / validation boundary.

In [ ]:
# Stage D: compare chronological train / validation boundaries
# 1. Freeze historical development block
historical_df = (
    benchmark_df.loc[
        benchmark_df["date"] < SELECTED_CUTOFF
    ]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

future_df = (
    benchmark_df.loc[
        benchmark_df["date"] >= SELECTED_CUTOFF
    ]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)
# 2. Candidate validation starts
CANDIDATE_VALIDATION_STARTS = [
    pd.Timestamp("2025-10-01", tz="UTC"),
    pd.Timestamp("2025-11-01", tz="UTC"),
    pd.Timestamp("2025-12-01", tz="UTC"),
]

rows = []

for validation_start in CANDIDATE_VALIDATION_STARTS:

    train_candidate = historical_df.loc[
        historical_df["date"] < validation_start
    ].copy()

    validation_candidate = historical_df.loc[
        historical_df["date"] >= validation_start
    ].copy()
    # Class support
    train_class_counts = (
        train_candidate["sentiment"]
        .value_counts()
        .reindex(
            ["negative", "neutral", "positive"],
            fill_value=0
        )
    )

    validation_class_counts = (
        validation_candidate["sentiment"]
        .value_counts()
        .reindex(
            ["negative", "neutral", "positive"],
            fill_value=0
        )
    )
    # Source support
    train_sources = set(
        train_candidate["channel_username"]
        .dropna()
        .unique()
    )

    validation_sources = set(
        validation_candidate["channel_username"]
        .dropna()
        .unique()
    )

    source_union = train_sources | validation_sources
    source_intersection = train_sources & validation_sources

    source_jaccard = (
        len(source_intersection) / len(source_union)
        if source_union else np.nan
    )
    # Validation calendar coverage
    validation_months = (
        validation_candidate["date"]
        .dt.strftime("%Y-%m")
        .nunique()
    )
    # Record candidate
    rows.append({
        "validation_start_utc":
            validation_start.isoformat(),

        "train_n":
            int(len(train_candidate)),

        "validation_n":
            int(len(validation_candidate)),

        "train_fraction_of_historical":
            float(
                len(train_candidate) /
                len(historical_df)
            ),

        "validation_fraction_of_historical":
            float(
                len(validation_candidate) /
                len(historical_df)
            ),

        "train_negative_n":
            int(train_class_counts["negative"]),

        "train_neutral_n":
            int(train_class_counts["neutral"]),

        "train_positive_n":
            int(train_class_counts["positive"]),

        "validation_negative_n":
            int(validation_class_counts["negative"]),

        "validation_neutral_n":
            int(validation_class_counts["neutral"]),

        "validation_positive_n":
            int(validation_class_counts["positive"]),

        "train_min_class_n":
            int(train_class_counts.min()),

        "validation_min_class_n":
            int(validation_class_counts.min()),

        "train_sources":
            int(len(train_sources)),

        "validation_sources":
            int(len(validation_sources)),

        "shared_sources":
            int(len(source_intersection)),

        "source_overlap_jaccard":
            float(source_jaccard),

        "validation_calendar_months":
            int(validation_months),
    })

train_validation_feasibility = pd.DataFrame(rows)

train_validation_feasibility[
    "all_train_classes_present"
] = (
    train_validation_feasibility[
        [
            "train_negative_n",
            "train_neutral_n",
            "train_positive_n",
        ]
    ].min(axis=1) > 0
)

train_validation_feasibility[
    "all_validation_classes_present"
] = (
    train_validation_feasibility[
        [
            "validation_negative_n",
            "validation_neutral_n",
            "validation_positive_n",
        ]
    ].min(axis=1) > 0
)

report_columns = [
    "validation_start_utc",
    "train_n",
    "validation_n",
    "train_fraction_of_historical",
    "validation_fraction_of_historical",
    "train_min_class_n",
    "validation_min_class_n",
    "validation_calendar_months",
    "train_sources",
    "validation_sources",
    "shared_sources",
    "source_overlap_jaccard",
]

print("=" * 120)
print("DAY 14 — STAGE D: TRAIN / VALIDATION FEASIBILITY")
print("=" * 120)

print(
    train_validation_feasibility[
        report_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nHistorical development block:")
print(
    f"Start : {historical_df['date'].min()}"
)
print(
    f"End   : {historical_df['date'].max()}"
)
print(
    f"N     : {len(historical_df):,}"
)

print("\nStrictly future block:")
print(
    f"Start : {future_df['date'].min()}"
)
print(
    f"End   : {future_df['date'].max()}"
)
print(
    f"N     : {len(future_df):,}"
)

print("=" * 120)

### Stage D Decision — Historical Train / Validation Boundary

The historical development block contains 19,398 observations before 2026-01-01.

The final chronological split is:

- **Historical Train:** 2025-06-03 21:06:16 UTC through 2025-10-31
- **Historical Validation:** 2025-11-01 through 2025-12-31
- **Strictly Future Evaluation:** begins 2026-01-01

The 2025-11-01 validation boundary is selected because it preserves:

- a substantial historical training block;
- two complete calendar months for validation;
- strong support for all three sentiment classes;
- high source overlap between train and validation.

The boundary is selected without using any model-performance information.

In [ ]:
# Stage D: freeze historical train / validation split
# Freeze validation boundary

VALIDATION_START = pd.Timestamp(
    "2025-11-01",
    tz="UTC"
)

# Historical cutoff already frozen at 2026-01-01 UTC
VALIDATION_END_EXCLUSIVE = SELECTED_CUTOFF

train_df = (
    benchmark_df.loc[
        benchmark_df["date"] < VALIDATION_START
    ]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

validation_df = (
    benchmark_df.loc[
        (benchmark_df["date"] >= VALIDATION_START)
        &
        (benchmark_df["date"] < SELECTED_CUTOFF)
    ]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

future_df = (
    benchmark_df.loc[
        benchmark_df["date"] >= SELECTED_CUTOFF
    ]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

checks = {
    "train_nonempty":
        len(train_df) > 0,

    "validation_nonempty":
        len(validation_df) > 0,

    "future_nonempty":
        len(future_df) > 0,

    "train_before_validation":
        train_df["date"].max()
        < validation_df["date"].min(),

    "validation_before_future":
        validation_df["date"].max()
        < future_df["date"].min(),

    "train_plus_validation_equals_historical":
        len(train_df) + len(validation_df)
        == len(historical_df),

    "all_splits_equal_benchmark":
        len(train_df)
        + len(validation_df)
        + len(future_df)
        == len(benchmark_df),

    "train_three_classes":
        set(train_df["sentiment"].unique())
        == {"negative", "neutral", "positive"},

    "validation_three_classes":
        set(validation_df["sentiment"].unique())
        == {"negative", "neutral", "positive"},

    "future_three_classes":
        set(future_df["sentiment"].unique())
        == {"negative", "neutral", "positive"},
}

def get_class_counts(df):
    return (
        df["sentiment"]
        .value_counts()
        .reindex(
            ["negative", "neutral", "positive"],
            fill_value=0
        )
    )

train_counts = get_class_counts(train_df)
validation_counts = get_class_counts(validation_df)
future_counts = get_class_counts(future_df)

print("=" * 96)
print("DAY 14 — STAGE D: CHRONOLOGICAL TRAIN / VALIDATION SPLIT")
print("=" * 96)

print("\nHistorical Train")
print("-" * 96)
print(f"N       : {len(train_df):,}")
print(f"Start   : {train_df['date'].min()}")
print(f"End     : {train_df['date'].max()}")
print(f"Sources : {train_df['channel_username'].nunique(dropna=True)}")
print(
    "Classes : "
    f"negative={train_counts['negative']:,}, "
    f"neutral={train_counts['neutral']:,}, "
    f"positive={train_counts['positive']:,}"
)

print("\nHistorical Validation")
print("-" * 96)
print(f"N       : {len(validation_df):,}")
print(f"Start   : {validation_df['date'].min()}")
print(f"End     : {validation_df['date'].max()}")
print(
    f"Months  : "
    f"{validation_df['date'].dt.strftime('%Y-%m').nunique()}"
)
print(
    f"Sources : "
    f"{validation_df['channel_username'].nunique(dropna=True)}"
)
print(
    "Classes : "
    f"negative={validation_counts['negative']:,}, "
    f"neutral={validation_counts['neutral']:,}, "
    f"positive={validation_counts['positive']:,}"
)

print("\nStrictly Future")
print("-" * 96)
print(f"N       : {len(future_df):,}")
print(f"Start   : {future_df['date'].min()}")
print(f"End     : {future_df['date'].max()}")
print(f"Sources : {future_df['channel_username'].nunique(dropna=True)}")
print(
    "Classes : "
    f"negative={future_counts['negative']:,}, "
    f"neutral={future_counts['neutral']:,}, "
    f"positive={future_counts['positive']:,}"
)

print("\nValidation checks:")
for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} — {name}")

if not all(checks.values()):
    failed = [
        name for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nStage D chronological split validation failed:\n"
        + "\n".join(f"- {name}" for name in failed)
    )

if len(train_df) != 13_539:
    raise AssertionError(
        f"Unexpected train N: {len(train_df):,}"
    )

if len(validation_df) != 5_859:
    raise AssertionError(
        f"Unexpected validation N: {len(validation_df):,}"
    )

if len(future_df) != 20_002:
    raise AssertionError(
        f"Unexpected future N: {len(future_df):,}"
    )

print("\n" + "-" * 96)
print(
    "STAGE D: PASS — HISTORICAL TRAIN / VALIDATION "
    "BOUNDARY FROZEN AT 2025-11-01 UTC"
)
print("=" * 96)

## Stage E — Define Strictly Future Evaluation Windows

All observations after the frozen historical development boundary are reserved for strictly future evaluation.

The default design uses calendar-month windows because they are interpretable, reproducible, and preserve chronology.

For each future window, record:

- exact start and end timestamps;
- sample size;
- class counts and proportions;
- source counts;
- temporal distance from the training endpoint;
- whether the window is a complete or partial calendar month.

The final June 2026 window must be explicitly marked as partial if retained.

In [ ]:
# Stage E: inspect strictly future calendar-month windows
# 1. Assign calendar-month labels
future_window_check = future_df.copy()

future_window_check["calendar_month"] = (
    future_window_check["date"]
    .dt.strftime("%Y-%m")
)

# 2. Aggregate each calendar month
rows = []

for month, month_df in future_window_check.groupby(
    "calendar_month",
    sort=True
):

    class_counts = (
        month_df["sentiment"]
        .value_counts()
        .reindex(
            ["negative", "neutral", "positive"],
            fill_value=0
        )
    )

    rows.append({
        "calendar_month":
            month,

        "start_timestamp":
            month_df["date"].min(),

        "end_timestamp":
            month_df["date"].max(),

        "n":
            int(len(month_df)),

        "negative_n":
            int(class_counts["negative"]),

        "neutral_n":
            int(class_counts["neutral"]),

        "positive_n":
            int(class_counts["positive"]),

        "min_class_n":
            int(class_counts.min()),

        "n_sources":
            int(
                month_df[
                    "channel_username"
                ].nunique(dropna=True)
            ),
    })

future_month_feasibility = pd.DataFrame(rows)
# 3. Identify complete vs partial month
benchmark_end_month = benchmark_df["date"].max().strftime("%Y-%m")

future_month_feasibility["is_partial_month"] = (
    future_month_feasibility["calendar_month"]
    == benchmark_end_month
)

# 4. Add simple adequacy check
future_month_feasibility["all_classes_present"] = (
    future_month_feasibility[
        ["negative_n", "neutral_n", "positive_n"]
    ].min(axis=1) > 0
)

future_month_feasibility["adequate_class_support"] = (
    future_month_feasibility["min_class_n"] >= 50
)

print("=" * 118)
print("DAY 14 — STAGE E: FUTURE CALENDAR-MONTH FEASIBILITY")
print("=" * 118)

print(
    future_month_feasibility.to_string(
        index=False
    )
)

print("\nSummary:")
print(
    f"Future observations : "
    f"{future_month_feasibility['n'].sum():,}"
)
print(
    f"Calendar windows    : "
    f"{len(future_month_feasibility)}"
)
print(
    f"Partial windows     : "
    f"{future_month_feasibility['is_partial_month'].sum()}"
)

print("=" * 118)

### Stage E Decision — Freeze Future Evaluation Windows

The strictly future evaluation period is partitioned into six fixed calendar windows:

- **F1:** January 2026
- **F2:** February 2026
- **F3:** March 2026
- **F4:** April 2026
- **F5:** May 2026
- **F6:** June 1–5, 2026 (**partial month**)

F6 is retained because it still contains 600 observations, all three sentiment classes, and adequate class support. However, it remains explicitly marked as a partial window and will be interpreted cautiously in temporal-trend analyses.

For each future window, temporal distance is recorded relative to both:

1. the end of the historical training block; and
2. the frozen historical development boundary.

This preserves the distinction between distance from the data used for LoRA fitting and distance from the full historical model-development period.

In [ ]:
# Stage E: freeze F1-F6 future evaluation windows
# 1. Assign fixed future-window IDs
MONTH_TO_WINDOW = {
    "2026-01": "F1",
    "2026-02": "F2",
    "2026-03": "F3",
    "2026-04": "F4",
    "2026-05": "F5",
    "2026-06": "F6",
}

future_df["calendar_month"] = (
    future_df["date"]
    .dt.strftime("%Y-%m")
)

future_df["future_window"] = (
    future_df["calendar_month"]
    .map(MONTH_TO_WINDOW)
)

if future_df["future_window"].isna().any():
    raise ValueError(
        "Some future observations were not assigned "
        "to a frozen F1-F6 window."
    )
# 2. Reference timestamps for temporal distance
TRAIN_END_TIMESTAMP = train_df["date"].max()
DEVELOPMENT_END_TIMESTAMP = validation_df["date"].max()

# 3. Build frozen future-window manifest

rows = []

for window_id in [
    "F1", "F2", "F3", "F4", "F5", "F6"
]:

    window_df = (
        future_df.loc[
            future_df["future_window"] == window_id
        ]
        .copy()
        .sort_values("date")
    )

    calendar_month = (
        window_df["calendar_month"].iloc[0]
    )

    class_counts = (
        window_df["sentiment"]
        .value_counts()
        .reindex(
            ["negative", "neutral", "positive"],
            fill_value=0
        )
    )

    class_props = (
        class_counts / len(window_df)
    )

    start_ts = window_df["date"].min()
    end_ts = window_df["date"].max()

    # Use window midpoint for trend analysis.
    midpoint_ts = (
        start_ts
        + (end_ts - start_ts) / 2
    )

    rows.append({
        "future_window":
            window_id,

        "calendar_month":
            calendar_month,

        "window_order":
            int(window_id[1:]),

        "start_timestamp_utc":
            start_ts.isoformat(),

        "end_timestamp_utc":
            end_ts.isoformat(),

        "midpoint_timestamp_utc":
            midpoint_ts.isoformat(),

        "n":
            int(len(window_df)),

        "negative_n":
            int(class_counts["negative"]),

        "neutral_n":
            int(class_counts["neutral"]),

        "positive_n":
            int(class_counts["positive"]),

        "negative_prop":
            float(class_props["negative"]),

        "neutral_prop":
            float(class_props["neutral"]),

        "positive_prop":
            float(class_props["positive"]),

        "min_class_n":
            int(class_counts.min()),

        "n_sources":
            int(
                window_df[
                    "channel_username"
                ].nunique(dropna=True)
            ),

        "is_partial_month":
            bool(window_id == "F6"),

        "days_from_train_end_to_start":
            float(
                (
                    start_ts
                    - TRAIN_END_TIMESTAMP
                ).total_seconds()
                / 86400
            ),

        "days_from_train_end_to_midpoint":
            float(
                (
                    midpoint_ts
                    - TRAIN_END_TIMESTAMP
                ).total_seconds()
                / 86400
            ),

        "days_from_development_end_to_start":
            float(
                (
                    start_ts
                    - DEVELOPMENT_END_TIMESTAMP
                ).total_seconds()
                / 86400
            ),

        "days_from_development_end_to_midpoint":
            float(
                (
                    midpoint_ts
                    - DEVELOPMENT_END_TIMESTAMP
                ).total_seconds()
                / 86400
            ),
    })

future_window_manifest = pd.DataFrame(rows)

# 4. Frozen-window validation check
checks = {
    "six_future_windows":
        len(future_window_manifest) == 6,

    "window_ids_correct":
        future_window_manifest[
            "future_window"
        ].tolist()
        == ["F1", "F2", "F3", "F4", "F5", "F6"],

    "future_n_preserved":
        future_window_manifest["n"].sum()
        == len(future_df)
        == 20_002,

    "all_classes_supported":
        (
            future_window_manifest[
                "min_class_n"
            ] >= 50
        ).all(),

    "only_f6_partial":
        (
            future_window_manifest[
                "is_partial_month"
            ].sum() == 1
            and
            bool(
                future_window_manifest.loc[
                    future_window_manifest[
                        "future_window"
                    ] == "F6",
                    "is_partial_month"
                ].iloc[0]
            )
        ),

    "strict_window_order":
        future_window_manifest[
            "window_order"
        ].is_monotonic_increasing,
}
# 5. Save formal artifact
future_manifest_path = (
    DAY14_DIR /
    "future_window_manifest.csv"
)

future_window_manifest.to_csv(
    future_manifest_path,
    index=False
)
# 6. Report
print("=" * 132)
print("DAY 14 — STAGE E: FROZEN FUTURE WINDOWS")
print("=" * 132)

report_columns = [
    "future_window",
    "calendar_month",
    "n",
    "negative_n",
    "neutral_n",
    "positive_n",
    "min_class_n",
    "n_sources",
    "is_partial_month",
    "days_from_train_end_to_midpoint",
    "days_from_development_end_to_midpoint",
]

print(
    future_window_manifest[
        report_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

print("\nValidation checks:")
for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} — {name}")

if not all(checks.values()):

    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nStage E future-window validation failed:\n"
        + "\n".join(f"- {name}" for name in failed)
    )

if not future_manifest_path.exists():
    raise RuntimeError(
        "future_window_manifest.csv was not saved."
    )

print(
    f"\nSaved:\n{future_manifest_path}"
)

print("\n" + "-" * 132)
print(
    "STAGE E: PASS — FUTURE WINDOWS F1-F6 FROZEN"
)
print("=" * 132)

### Chronological Split Specification

The complete temporal benchmark structure is now frozen:

Historical Train  
→ Historical Validation  
→ F1 → F2 → F3 → F4 → F5 → F6

The split specification records the exact temporal boundaries, sample counts, class support, and future-window definitions.

No future observation may be used for:

- LoRA fitting;
- checkpoint selection;
- early stopping;
- hyperparameter tuning;
- threshold tuning;
- model-selection decisions.

In [ ]:
# Save complete chronological split specification
# Helper for split summaries
def summarize_split(df, split_name):
    class_counts = (
        df["sentiment"]
        .value_counts()
        .reindex(
            ["negative", "neutral", "positive"],
            fill_value=0
        )
    )

    return {
        "split_name": split_name,
        "n_observations": int(len(df)),
        "start_timestamp_utc": df["date"].min().isoformat(),
        "end_timestamp_utc": df["date"].max().isoformat(),
        "negative_n": int(class_counts["negative"]),
        "neutral_n": int(class_counts["neutral"]),
        "positive_n": int(class_counts["positive"]),
        "n_sources": int(
            df["channel_username"]
            .nunique(dropna=True)
        )
    }

# Core split summaries
train_spec = summarize_split(
    train_df,
    "historical_train"
)

validation_spec = summarize_split(
    validation_df,
    "historical_validation"
)

future_spec = summarize_split(
    future_df,
    "strictly_future_total"
)

# Future-window specifications

future_windows_spec = []

for _, row in future_window_manifest.iterrows():

    future_windows_spec.append({
        "future_window":
            row["future_window"],

        "calendar_month":
            row["calendar_month"],

        "window_order":
            int(row["window_order"]),

        "start_timestamp_utc":
            row["start_timestamp_utc"],

        "end_timestamp_utc":
            row["end_timestamp_utc"],

        "midpoint_timestamp_utc":
            row["midpoint_timestamp_utc"],

        "n_observations":
            int(row["n"]),

        "negative_n":
            int(row["negative_n"]),

        "neutral_n":
            int(row["neutral_n"]),

        "positive_n":
            int(row["positive_n"]),

        "n_sources":
            int(row["n_sources"]),

        "is_partial_month":
            bool(row["is_partial_month"]),

        "days_from_train_end_to_midpoint":
            float(
                row[
                    "days_from_train_end_to_midpoint"
                ]
            ),

        "days_from_development_end_to_midpoint":
            float(
                row[
                    "days_from_development_end_to_midpoint"
                ]
            )
    })
# Build chronological split specification
chronological_split_spec = {
    "day": 14,
    "artifact": "chronological_split_spec",

    "benchmark_population": {
        "dataset": "Dataset B",
        "n_observations": int(len(benchmark_df)),
        "start_timestamp_utc":
            benchmark_df["date"].min().isoformat(),
        "end_timestamp_utc":
            benchmark_df["date"].max().isoformat()
    },

    "historical_cutoff": {
        "cutoff_timestamp_utc":
            SELECTED_CUTOFF.isoformat(),

        "selection_basis":
            (
                "Pre-model calendar/sample-support feasibility; "
                "no model performance used."
            )
    },

    "historical_train": train_spec,

    "historical_validation": validation_spec,

    "strictly_future_total": future_spec,

    "future_windows": future_windows_spec,

    "ordering_rule": (
        "historical_train < historical_validation "
        "< all future evaluation windows"
    ),

    "future_data_prohibition": {
        "training": True,
        "checkpoint_selection": True,
        "early_stopping": True,
        "hyperparameter_selection": True,
        "threshold_tuning": True
    },

    "partial_window_rule": {
        "F6_is_partial": True,
        "interpretation_rule":
            (
                "F6 is retained because class support is adequate, "
                "but temporal-trend interpretation must explicitly "
                "acknowledge its partial-month status."
            )
    },

    "freeze_status": "LOCKED"
}
# Save artifact
split_spec_path = (
    DAY14_DIR /
    "chronological_split_spec.json"
)

with open(
    split_spec_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        chronological_split_spec,
        f,
        indent=2,
        ensure_ascii=False
    )
# Cross-checks
checks = {
    "train_n":
        train_spec["n_observations"]
        == 13_539,

    "validation_n":
        validation_spec["n_observations"]
        == 5_859,

    "future_n":
        future_spec["n_observations"]
        == 20_002,

    "benchmark_total":
        (
            train_spec["n_observations"]
            + validation_spec["n_observations"]
            + future_spec["n_observations"]
            == 39_400
        ),

    "six_future_windows":
        len(future_windows_spec) == 6,

    "future_window_total":
        sum(
            w["n_observations"]
            for w in future_windows_spec
        )
        == 20_002,

    "chronology_train_validation":
        train_df["date"].max()
        < validation_df["date"].min(),

    "chronology_validation_future":
        validation_df["date"].max()
        < future_df["date"].min(),

    "artifact_saved":
        split_spec_path.exists()
}
print("=" * 96)
print("DAY 14 — CHRONOLOGICAL SPLIT SPECIFICATION")
print("=" * 96)

print("\nFrozen benchmark structure:")
print(
    f"Train      : {len(train_df):,} "
    f"| {train_df['date'].min()} "
    f"-> {train_df['date'].max()}"
)

print(
    f"Validation : {len(validation_df):,} "
    f"| {validation_df['date'].min()} "
    f"-> {validation_df['date'].max()}"
)

for _, row in future_window_manifest.iterrows():
    partial = (
        " [PARTIAL]"
        if row["is_partial_month"]
        else ""
    )

    print(
        f"{row['future_window']:<10}: "
        f"{int(row['n']):,} "
        f"| {row['start_timestamp_utc']} "
        f"-> {row['end_timestamp_utc']}"
        f"{partial}"
    )

print("\nValidation checks:")

for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} — {name}")

if not all(checks.values()):
    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nChronological split specification failed:\n"
        + "\n".join(
            f"- {name}"
            for name in failed
        )
    )

print(
    f"\nSaved:\n{split_spec_path}"
)

print("\n" + "-" * 96)
print(
    "PASS — COMPLETE CHRONOLOGICAL BENCHMARK "
    "STRUCTURE FROZEN"
)
print("=" * 96)

## Stage F — Freeze the Dataset-B LoRA Training Protocol

The Day 15 model will preserve the core Day 04 FinGPT configuration while replacing the original Dataset-A train/validation data with the frozen chronological Dataset-B historical train/validation blocks.

### Preserved from Day 04

- Base model: `meta-llama/Meta-Llama-3-8B`
- 4-bit NF4 quantization
- LoRA on `q_proj` and `v_proj`
- LoRA rank = 8
- LoRA alpha = 32
- LoRA dropout = 0.1
- Maximum sequence length = 512
- Target-only causal-LM masking
- Train batch size = 1
- Validation batch size = 1
- Gradient accumulation = 8
- Learning rate = 1e-4
- Weight decay = 0.01
- Warmup steps = 10
- FP16 training
- Maximum training budget = 200 steps
- Greedy label generation with `max_new_tokens=4`

### Day 14 protocol refinement

Day 04 trained to a fixed final 200-step adapter and did not load the best validation checkpoint.

For the chronological benchmark, checkpoint selection is frozen in advance:

- train for at most 200 steps;
- save checkpoints at steps 50, 100, 150, and 200;
- select the checkpoint with the **lowest historical-validation loss**;
- future windows F1–F6 are completely prohibited from checkpoint selection;
- no early stopping, hyperparameter search, threshold tuning, or future adaptation is permitted.

This change makes model selection explicitly historical-only while preserving the original Day 04 training budget and architecture.

In [ ]:
# Stage F: freeze Dataset-B historical LoRA training protocol

# 1. Frozen Day 15 LoRA specification

lora_training_protocol = {
    "day": 14,
    "artifact": "lora_training_protocol",
    "purpose": (
        "Pre-specify the Day 15 Dataset-B historical LoRA training "
        "recipe before observing any new Dataset-B-trained model performance."
    ),
    # Data protocol

    "data_protocol": {
        "training_split": "historical_train",
        "training_n": int(len(train_df)),

        "validation_split": "historical_validation",
        "validation_n": int(len(validation_df)),

        "training_start_utc":
            train_df["date"].min().isoformat(),

        "training_end_utc":
            train_df["date"].max().isoformat(),

        "validation_start_utc":
            validation_df["date"].min().isoformat(),

        "validation_end_utc":
            validation_df["date"].max().isoformat(),

        "future_windows_prohibited": [
            "F1", "F2", "F3", "F4", "F5", "F6"
        ],

        "future_data_may_not_affect": [
            "training",
            "checkpoint_selection",
            "early_stopping",
            "hyperparameter_selection",
            "threshold_tuning",
            "prompt_selection",
            "label_mapping"
        ]
    },

    # Base model / tokenizer
    "base_model": {
        "model_name":
            "meta-llama/Meta-Llama-3-8B",

        "model_family":
            "Meta-Llama-3",

        "task":
            "causal_language_modeling",

        "trust_remote_code":
            True,

        "device_map":
            "auto"
    },
    # Quantization
    "quantization": {
        "enabled": True,
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_use_double_quant": True,
        "bnb_4bit_compute_dtype": "float16"
    },
    # LoRA
    "lora": {
        "task_type": "CAUSAL_LM",
        "inference_mode": False,
        "r": 8,
        "lora_alpha": 32,
        "lora_dropout": 0.1,
        "target_modules": [
            "q_proj",
            "v_proj"
        ],
        "bias": "none"
    },
    # Sequence construction

    "sequence_protocol": {
        "max_seq_length": 512,

        "prompt_template": (
            "Instruction:{instruction}\\n"
            "Input:{input}\\n"
            "Answer: "
        ),

        "target":
            "sentiment label",

        "target_labels": [
            "negative",
            "neutral",
            "positive"
        ],

        "append_eos_token":
            True,

        "loss_masking":
            "mask complete prompt with -100; compute causal-LM loss only on target/EOS tokens",

        "dynamic_batch_padding":
            True,

        "padding_label_value":
            -100,

        "training_pad_token":
            "eos_token"
    },
# Training hyperparameters
    "training": {
        "max_steps": 200,

        "per_device_train_batch_size": 1,
        "per_device_eval_batch_size": 1,

        "gradient_accumulation_steps": 8,

        "effective_train_batch_size":
            8,

        "learning_rate":
            1e-4,

        "weight_decay":
            0.01,

        "warmup_steps":
            10,

        "fp16":
            True,

        "gradient_checkpointing":
            True,

        "use_cache_during_training":
            False,

        "logging_steps":
            10,

        "evaluation_strategy":
            "steps",

        "eval_steps":
            25,

        "save_strategy":
            "steps",

        "save_steps":
            50,

        "save_total_limit":
            4,

        "report_to":
            "none",

        "remove_unused_columns":
            False,

        # Transformers default is 42; Day 14 now freezes it explicitly.
        "seed":
            42,

        "data_seed":
            42
    },

    "checkpoint_selection": {
        "candidate_steps": [
            50,
            100,
            150,
            200
        ],

        "selection_dataset":
            "historical_validation only",

        "selection_metric":
            "eval_loss",

        "selection_direction":
            "minimize",

        "early_stopping":
            False,

        "maximum_training_budget_steps":
            200,

        "tie_break_rule":
            (
                "If multiple checkpoints have numerically identical "
                "validation loss at stored precision, select the earliest step."
            ),

        "future_window_metrics_allowed":
            False
    },


    "inference": {
        "model_mode":
            "eval",

        "tokenizer_padding_side":
            "left",

        "pad_token":
            "eos_token",

        "generation": {
            "max_new_tokens": 4,
            "do_sample": False,
            "pad_token_id": "eos_token_id",
            "eos_token_id": "eos_token_id"
        },

        "prediction_postprocessing": [
            "decode generated tokens only",
            "skip special tokens",
            "strip whitespace",
            "lowercase"
        ],

        "valid_predictions": [
            "negative",
            "neutral",
            "positive"
        ]
    },

    "day04_comparability": {
        "core_architecture_preserved":
            True,

        "quantization_preserved":
            True,

        "lora_configuration_preserved":
            True,

        "sequence_length_preserved":
            True,

        "optimization_hyperparameters_preserved":
            True,

        "maximum_step_budget_preserved":
            True,

        "generation_rule_preserved":
            True,

        "intentional_protocol_change": (
            "Day 04 retained the final 200-step model with "
            "load_best_model_at_end=False. "
            "Day 15 will instead select among pre-specified saved "
            "checkpoints using historical-validation eval_loss only."
        )
    },

    "freeze_status":
        "LOCKED_BEFORE_DAY15"
}


checks = {
    "correct_base_model":
        lora_training_protocol[
            "base_model"
        ]["model_name"]
        == "meta-llama/Meta-Llama-3-8B",

    "four_bit_nf4":
        (
            lora_training_protocol[
                "quantization"
            ]["load_in_4bit"]
            and
            lora_training_protocol[
                "quantization"
            ]["bnb_4bit_quant_type"] == "nf4"
        ),

    "lora_rank_8":
        lora_training_protocol[
            "lora"
        ]["r"] == 8,

    "lora_alpha_32":
        lora_training_protocol[
            "lora"
        ]["lora_alpha"] == 32,

    "lora_targets_qv":
        lora_training_protocol[
            "lora"
        ]["target_modules"]
        == ["q_proj", "v_proj"],

    "max_seq_length_512":
        lora_training_protocol[
            "sequence_protocol"
        ]["max_seq_length"] == 512,

    "training_budget_200":
        lora_training_protocol[
            "training"
        ]["max_steps"] == 200,

    "learning_rate_1e4":
        lora_training_protocol[
            "training"
        ]["learning_rate"] == 1e-4,

    "gradient_accumulation_8":
        lora_training_protocol[
            "training"
        ]["gradient_accumulation_steps"] == 8,

    "seed_frozen":
        lora_training_protocol[
            "training"
        ]["seed"] == 42,

    "historical_validation_checkpoint_only":
        lora_training_protocol[
            "checkpoint_selection"
        ]["selection_dataset"]
        == "historical_validation only",

    "future_windows_prohibited":
        len(
            lora_training_protocol[
                "data_protocol"
            ]["future_windows_prohibited"]
        ) == 6,

    "greedy_generation":
        (
            lora_training_protocol[
                "inference"
            ]["generation"]["do_sample"]
            is False
        ),

    "four_generation_tokens":
        lora_training_protocol[
            "inference"
        ]["generation"]["max_new_tokens"] == 4
}
lora_protocol_path = (
    DAY14_DIR /
    "lora_training_protocol.json"
)

with open(
    lora_protocol_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        lora_training_protocol,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 100)
print("DAY 14 — STAGE F: FROZEN LoRA TRAINING PROTOCOL")
print("=" * 100)

print("\nData:")
print(
    f"Historical train      : "
    f"{len(train_df):,}"
)
print(
    f"Historical validation : "
    f"{len(validation_df):,}"
)
print(
    "Future windows        : "
    "F1-F6 PROHIBITED from training/model selection"
)

print("\nModel:")
print(
    "Base model            : "
    "meta-llama/Meta-Llama-3-8B"
)
print(
    "Quantization          : "
    "4-bit NF4 + double quantization"
)
print(
    "LoRA                  : "
    "r=8, alpha=32, dropout=0.1, q_proj/v_proj"
)

print("\nTraining:")
print("Max steps             : 200")
print("Train batch size      : 1")
print("Gradient accumulation : 8")
print("Effective batch size  : 8")
print("Learning rate         : 1e-4")
print("Weight decay          : 0.01")
print("Warmup steps          : 10")
print("Seed                   : 42")

print("\nCheckpoint rule:")
print(
    "Candidates            : "
    "50, 100, 150, 200"
)
print(
    "Selection metric      : "
    "historical-validation eval_loss"
)
print(
    "Direction             : minimize"
)
print(
    "Early stopping        : disabled"
)

print("\nInference:")
print(
    "Decoding              : "
    "greedy"
)
print(
    "max_new_tokens        : 4"
)

print("\nValidation checks:")
for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} — {name}")

if not all(checks.values()):
    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nStage F protocol validation failed:\n"
        + "\n".join(
            f"- {name}"
            for name in failed
        )
    )

if not lora_protocol_path.exists():
    raise RuntimeError(
        "lora_training_protocol.json was not saved."
    )

print(
    f"\nSaved:\n{lora_protocol_path}"
)

print("\n" + "-" * 100)
print(
    "STAGE F: PASS — DAY 15 LoRA TRAINING "
    "PROTOCOL FROZEN"
)
print("=" * 100)

## Stage G — Freeze Evaluation Metrics

The same evaluation protocol will be applied to:

- historical validation;
- F1;
- F2;
- F3;
- F4;
- F5;
- F6.

### Primary metric

**Macro-F1**

Macro-F1 is the primary temporal-generalization metric because Dataset B has unequal class frequencies and prior project stages identified important class-specific failure modes.

### Secondary metrics

- Accuracy
- Weighted F1

### Class-specific diagnostics

- Precision by class
- Recall by class
- F1 by class
- Support by class

### Failure-mode diagnostics

The full 3 × 3 confusion matrix will be reported using the fixed label order:

`negative → neutral → positive`

Special attention will be given to directional error transitions, particularly:

- negative → neutral
- negative → positive
- neutral → negative
- neutral → positive
- positive → negative
- positive → neutral

### Prediction-behavior diagnostics

For every evaluation split/window, report:

- predicted negative proportion;
- predicted neutral proportion;
- predicted positive proportion;
- invalid prediction count and rate.

### Invalid-generation rule

After decoding, predictions are stripped and converted to lowercase.

Only exact matches to:

- `negative`
- `neutral`
- `positive`

are treated as valid sentiment predictions.

Any other generated output is recorded as `invalid` and counted as an incorrect prediction. No post-hoc semantic mapping or manual correction is allowed.

In [ ]:
# Stage G: freeze evaluation metrics and scoring rules

LABELS = [
    "negative",
    "neutral",
    "positive"
]

PRIMARY_METRIC = "macro_f1"

SECONDARY_METRICS = [
    "accuracy",
    "weighted_f1"
]

# Define complete frozen evaluation protocol
evaluation_protocol = {
    "day": 14,
    "artifact": "evaluation_protocol",

    "evaluation_scope": {
        "historical_validation": True,
        "future_windows": [
            "F1",
            "F2",
            "F3",
            "F4",
            "F5",
            "F6"
        ],

        "same_metric_implementation_for_all_splits":
            True
    },

    # Label schema

    "label_schema": {
        "labels": LABELS,
        "label_order": LABELS,

        "true_label_column":
            "sentiment",

        "prediction_column":
            "prediction",

        "case_normalization":
            "lowercase",

        "whitespace_normalization":
            "strip leading/trailing whitespace"
    },
    # Primary / secondary metrics
    "metrics": {
        "primary": {
            "name":
                PRIMARY_METRIC,

            "definition":
                "unweighted mean of per-class F1 scores",

            "reason":
                (
                    "Primary temporal-generalization metric because "
                    "class frequencies are unequal and class-specific "
                    "failure modes are scientifically important."
                )
        },

        "secondary": [
            {
                "name": "accuracy",
                "role": "overall predictive correctness"
            },
            {
                "name": "weighted_f1",
                "role": (
                    "class-frequency-weighted predictive performance"
                )
            }
        ],

        "class_specific": [
            "precision",
            "recall",
            "f1",
            "support"
        ]
    },

    # Confusion / failure-mode analysis

    "confusion_analysis": {
        "matrix_label_order":
            LABELS,

        "report_raw_counts":
            True,

        "report_row_normalized_rates":
            True,

        "directional_transitions": [
            "negative->neutral",
            "negative->positive",
            "neutral->negative",
            "neutral->positive",
            "positive->negative",
            "positive->neutral"
        ],

        "priority_failure_mode":
            "negative->neutral",

        "transition_denominator":
            "number of observations in the corresponding true class"
    },

    # Prediction-distribution diagnostics

    "prediction_behavior": {
        "report_predicted_class_counts":
            True,

        "report_predicted_class_proportions":
            True,

        "classes":
            LABELS,

        "purpose":
            (
                "Detect systematic changes such as increased "
                "neutral prediction tendency over time."
            )
    },

    # Invalid-generation handling
    "invalid_prediction_rule": {
        "valid_exact_predictions":
            LABELS,

        "postprocessing": [
            "decode generated tokens only",
            "skip special tokens",
            "strip whitespace",
            "lowercase"
        ],

        "semantic_remapping":
            False,

        "manual_correction":
            False,

        "invalid_label":
            "invalid",

        "invalid_predictions_count_as_errors":
            True,

        "report_invalid_count":
            True,

        "report_invalid_rate":
            True
    },

    # Temporal comparison rules
    "temporal_comparisons": {
        "validation_is_historical_baseline":
            True,

        "report_each_future_window_independently":
            True,

        "compute_change_vs_historical_validation":
            True,

        "compute_change_vs_F1":
            True,

        "primary_change_metric":
            "macro_f1"
    },

    # Partial-window handling

    "partial_window_rule": {
        "F6_retained":
            True,

        "F6_is_partial_month":
            True,

        "same_metrics_as_other_windows":
            True,

        "interpretation_caution_required":
            True
    },

    "freeze_status":
        "LOCKED_BEFORE_MODEL_EVALUATION"
}

# Protocol validation checks
checks = {
    "primary_macro_f1":
        evaluation_protocol[
            "metrics"
        ]["primary"]["name"]
        == "macro_f1",

    "secondary_accuracy":
        any(
            x["name"] == "accuracy"
            for x in evaluation_protocol[
                "metrics"
            ]["secondary"]
        ),

    "secondary_weighted_f1":
        any(
            x["name"] == "weighted_f1"
            for x in evaluation_protocol[
                "metrics"
            ]["secondary"]
        ),

    "three_class_order_frozen":
        evaluation_protocol[
            "label_schema"
        ]["label_order"]
        == [
            "negative",
            "neutral",
            "positive"
        ],

    "class_specific_metrics_present":
        set(
            evaluation_protocol[
                "metrics"
            ]["class_specific"]
        )
        == {
            "precision",
            "recall",
            "f1",
            "support"
        },

    "negative_to_neutral_tracked":
        "negative->neutral"
        in evaluation_protocol[
            "confusion_analysis"
        ]["directional_transitions"],

    "prediction_distribution_tracked":
        evaluation_protocol[
            "prediction_behavior"
        ]["report_predicted_class_proportions"]
        is True,

    "invalid_predictions_are_errors":
        evaluation_protocol[
            "invalid_prediction_rule"
        ]["invalid_predictions_count_as_errors"]
        is True,

    "no_semantic_remapping":
        evaluation_protocol[
            "invalid_prediction_rule"
        ]["semantic_remapping"]
        is False,

    "future_windows_complete":
        evaluation_protocol[
            "evaluation_scope"
        ]["future_windows"]
        == [
            "F1",
            "F2",
            "F3",
            "F4",
            "F5",
            "F6"
        ],

    "comparison_vs_validation":
        evaluation_protocol[
            "temporal_comparisons"
        ]["compute_change_vs_historical_validation"]
        is True,

    "comparison_vs_f1":
        evaluation_protocol[
            "temporal_comparisons"
        ]["compute_change_vs_F1"]
        is True
}
# 3. Save artifact
evaluation_protocol_path = (
    DAY14_DIR /
    "evaluation_protocol.json"
)

with open(
    evaluation_protocol_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evaluation_protocol,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 100)
print("DAY 14 — STAGE G: FROZEN EVALUATION PROTOCOL")
print("=" * 100)

print("\nPrimary metric:")
print("  Macro-F1")

print("\nSecondary metrics:")
print("  Accuracy")
print("  Weighted F1")

print("\nClass-specific metrics:")
print("  Precision / Recall / F1 / Support")

print("\nConfusion analysis:")
print("  Label order          : negative, neutral, positive")
print("  Raw matrix           : YES")
print("  Row-normalized rates : YES")
print("  Priority transition  : negative -> neutral")

print("\nPrediction behavior:")
print("  Class counts         : YES")
print("  Class proportions    : YES")
print("  Invalid predictions  : explicitly tracked")

print("\nInvalid-generation rule:")
print(
    "  Valid outputs        : "
    "negative / neutral / positive"
)
print(
    "  Other outputs        : "
    "invalid + counted as error"
)
print(
    "  Semantic remapping   : NO"
)

print("\nTemporal comparisons:")
print(
    "  Each F window independently        : YES"
)
print(
    "  Change vs historical validation    : YES"
)
print(
    "  Change vs F1                       : YES"
)

print("\nValidation checks:")

for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} — {name}")

if not all(checks.values()):
    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nStage G evaluation protocol failed:\n"
        + "\n".join(
            f"- {name}"
            for name in failed
        )
    )

if not evaluation_protocol_path.exists():
    raise RuntimeError(
        "evaluation_protocol.json was not saved."
    )

print(
    f"\nSaved:\n{evaluation_protocol_path}"
)

print("\n" + "-" * 100)
print(
    "STAGE G: PASS — EVALUATION METRICS "
    "AND SCORING RULES FROZEN"
)
print("=" * 100)

## Stage H — Pre-specify Temporal-Degradation Analysis

Temporal degradation is not required to be monotonic.

The frozen analysis will distinguish:

1. **window-specific performance**
2. **change relative to historical validation**
3. **change relative to F1**
4. **long-run temporal trend**
5. **localized regime changes**
6. **class-specific degradation mechanisms**

### Primary temporal outcome

Macro-F1 is the primary outcome for temporal degradation.

For each future window F1–F6, report:

- Macro-F1
- Accuracy
- Weighted F1
- per-class recall
- negative → neutral transition rate
- predicted label proportions
- invalid prediction rate

### Relative changes

For every future window:

- compute change relative to historical validation;
- compute change relative to F1.

### Temporal trend

Two frozen time predictors will be used:

- future-window order: F1 = 1, ..., F6 = 6;
- days from historical-development endpoint to future-window midpoint.

The primary trend analysis will fit a simple linear trend of Macro-F1 against temporal distance.

The trend coefficient is descriptive rather than a causal effect.

### Non-monotonicity rule

A temporary recovery does not invalidate temporal instability.

Therefore:

- no monotonicity constraint is imposed;
- localized drops and recoveries are reported explicitly;
- the worst future window is reported descriptively, not used to redefine the benchmark.

### Mechanism analysis

Class-specific recall and confusion transitions will be tracked across windows, with particular attention to negative recall and negative → neutral errors.

In [ ]:
# Stage H: freeze temporal-degradation analysis protocol
temporal_analysis_protocol = {
    "day": 14,
    "artifact": "temporal_analysis_protocol",

    # --------------------------------------------------------
    # Analysis scope
    # --------------------------------------------------------

    "analysis_scope": {
        "historical_baseline":
            "historical_validation",

        "future_windows": [
            "F1", "F2", "F3", "F4", "F5", "F6"
        ],

        "analyze_each_future_window_independently":
            True,

        "monotonic_degradation_required":
            False
    },

    # --------------------------------------------------------
    # Primary outcome
    # --------------------------------------------------------

    "primary_temporal_outcome": {
        "metric":
            "macro_f1",

        "role":
            "primary temporal-generalization outcome"
    },

    # --------------------------------------------------------
    # Secondary tracked outcomes
    # --------------------------------------------------------

    "tracked_outcomes": [
        "accuracy",
        "weighted_f1",

        "negative_recall",
        "neutral_recall",
        "positive_recall",

        "negative_to_neutral_rate",
        "negative_to_positive_rate",
        "neutral_to_negative_rate",
        "neutral_to_positive_rate",
        "positive_to_negative_rate",
        "positive_to_neutral_rate",

        "predicted_negative_proportion",
        "predicted_neutral_proportion",
        "predicted_positive_proportion",

        "invalid_prediction_rate"
    ],

    # --------------------------------------------------------
    # Relative-change analyses
    # --------------------------------------------------------

    "relative_change_analysis": {
        "vs_historical_validation": {
            "enabled": True,

            "metrics": [
                "macro_f1",
                "accuracy",
                "weighted_f1",
                "negative_recall",
                "neutral_recall",
                "positive_recall",
                "negative_to_neutral_rate"
            ],

            "formula":
                "future_window_metric - historical_validation_metric"
        },

        "vs_F1": {
            "enabled": True,

            "applicable_windows": [
                "F2", "F3", "F4", "F5", "F6"
            ],

            "metrics": [
                "macro_f1",
                "accuracy",
                "weighted_f1",
                "negative_recall",
                "neutral_recall",
                "positive_recall",
                "negative_to_neutral_rate"
            ],

            "formula":
                "future_window_metric - F1_metric"
        }
    },

    # --------------------------------------------------------
    # Temporal predictors
    # --------------------------------------------------------

    "temporal_predictors": {
        "window_order": {
            "values": {
                "F1": 1,
                "F2": 2,
                "F3": 3,
                "F4": 4,
                "F5": 5,
                "F6": 6
            },

            "role":
                "ordinal descriptive time index"
        },

        "continuous_time": {
            "variable":
                "days_from_development_end_to_midpoint",

            "source":
                "future_window_manifest.csv",

            "role":
                "primary continuous temporal-distance predictor"
        }
    },

    # --------------------------------------------------------
    # Trend model
    # --------------------------------------------------------

    "trend_analysis": {
        "primary_model": {
            "response":
                "macro_f1",

            "predictor":
                "days_from_development_end_to_midpoint",

            "model":
                "ordinary least squares linear trend",

            "interpretation":
                (
                    "Descriptive association between temporal distance "
                    "and future-window Macro-F1; not interpreted causally."
                )
        },

        "secondary_model": {
            "response":
                "macro_f1",

            "predictor":
                "window_order",

            "model":
                "ordinary least squares linear trend"
        },

        "report": [
            "slope",
            "intercept",
            "R_squared"
        ],

        "small_number_of_windows_caution":
            (
                "Trend estimates are descriptive because only six "
                "future windows are available."
            )
    },

    # --------------------------------------------------------
    # Localized regime-shift analysis
    # --------------------------------------------------------

    "localized_regime_analysis": {
        "enabled":
            True,

        "adjacent_window_changes":
            True,

        "primary_adjacent_metric":
            "macro_f1",

        "additional_adjacent_metrics": [
            "negative_recall",
            "negative_to_neutral_rate"
        ],

        "largest_drop_rule":
            "identify the most negative adjacent-window change",

        "largest_recovery_rule":
            "identify the most positive adjacent-window change",

        "interpretation_rule":
            (
                "Localized deterioration or recovery is reported "
                "without redefining the temporal benchmark."
            )
    },

    # --------------------------------------------------------
    # Failure mechanism analysis
    # --------------------------------------------------------

    "failure_mechanism_analysis": {
        "priority_class":
            "negative",

        "priority_recall_metric":
            "negative_recall",

        "priority_transition":
            "negative_to_neutral_rate",

        "additional_class_recalls": [
            "neutral_recall",
            "positive_recall"
        ],

        "prediction_distribution_tracking":
            True
    },

    # --------------------------------------------------------
    # Partial window handling
    # --------------------------------------------------------

    "partial_window_handling": {
        "F6_included_in_primary_tables":
            True,

        "F6_included_in_trend_model":
            True,

        "F6_partial_status_must_be_reported":
            True,

        "sensitivity_requirement":
            (
                "If F6 materially changes the fitted temporal slope, "
                "report an additional sensitivity trend using F1-F5 "
                "without replacing the pre-specified primary analysis."
            )
    },

    # --------------------------------------------------------
    # Prohibited post-hoc behavior
    # --------------------------------------------------------

    "prohibited_posthoc_changes": [
        "dropping a future window because performance is inconvenient",
        "changing the primary metric after observing future results",
        "redefining temporal cutoffs using performance",
        "requiring monotonic degradation after seeing results",
        "excluding F6 solely because its performance differs from other windows",
        "using the worst-performing window to redefine the benchmark"
    ],

    "freeze_status":
        "LOCKED_BEFORE_FUTURE_INFERENCE"
}

checks = {
    "primary_metric_macro_f1":
        temporal_analysis_protocol[
            "primary_temporal_outcome"
        ]["metric"] == "macro_f1",

    "all_six_windows":
        temporal_analysis_protocol[
            "analysis_scope"
        ]["future_windows"]
        == [
            "F1", "F2", "F3",
            "F4", "F5", "F6"
        ],

    "non_monotonic_allowed":
        temporal_analysis_protocol[
            "analysis_scope"
        ]["monotonic_degradation_required"]
        is False,

    "validation_baseline_enabled":
        temporal_analysis_protocol[
            "relative_change_analysis"
        ]["vs_historical_validation"]["enabled"]
        is True,

    "f1_baseline_enabled":
        temporal_analysis_protocol[
            "relative_change_analysis"
        ]["vs_F1"]["enabled"]
        is True,

    "continuous_time_frozen":
        temporal_analysis_protocol[
            "temporal_predictors"
        ]["continuous_time"]["variable"]
        == "days_from_development_end_to_midpoint",

    "linear_trend_frozen":
        temporal_analysis_protocol[
            "trend_analysis"
        ]["primary_model"]["model"]
        == "ordinary least squares linear trend",

    "negative_recall_tracked":
        "negative_recall"
        in temporal_analysis_protocol[
            "tracked_outcomes"
        ],

    "negative_to_neutral_tracked":
        temporal_analysis_protocol[
            "failure_mechanism_analysis"
        ]["priority_transition"]
        == "negative_to_neutral_rate",

    "localized_changes_enabled":
        temporal_analysis_protocol[
            "localized_regime_analysis"
        ]["enabled"]
        is True,

    "f6_retained":
        temporal_analysis_protocol[
            "partial_window_handling"
        ]["F6_included_in_primary_tables"]
        is True,

    "f6_sensitivity_prespecified":
        "F1-F5"
        in temporal_analysis_protocol[
            "partial_window_handling"
        ]["sensitivity_requirement"]
}

temporal_analysis_path = (
    DAY14_DIR /
    "temporal_analysis_protocol.json"
)

with open(
    temporal_analysis_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        temporal_analysis_protocol,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 104)
print("DAY 14 — STAGE H: TEMPORAL-DEGRADATION ANALYSIS PROTOCOL")
print("=" * 104)

print("\nPrimary temporal outcome:")
print("  Macro-F1")

print("\nTemporal baselines:")
print("  Historical validation")
print("  F1")

print("\nTemporal predictors:")
print("  Window order: F1=1 ... F6=6")
print(
    "  Continuous: days from historical-development "
    "endpoint to window midpoint"
)

print("\nPrimary trend:")
print(
    "  Macro-F1 ~ temporal distance "
    "(OLS descriptive linear trend)"
)

print("\nLocalized regime analysis:")
print("  Adjacent-window Macro-F1 changes")
print("  Adjacent negative-recall changes")
print("  Adjacent negative->neutral changes")

print("\nNon-monotonicity:")
print("  Monotonic deterioration required : NO")
print("  Temporary recovery allowed       : YES")

print("\nF6 handling:")
print("  Included in primary analysis     : YES")
print("  Explicitly marked partial        : YES")
print(
    "  F1-F5 sensitivity trend          : "
    "pre-specified if F6 materially changes slope"
)

print("\nValidation checks:")

for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} — {name}")

if not all(checks.values()):
    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nStage H temporal-analysis protocol failed:\n"
        + "\n".join(
            f"- {name}"
            for name in failed
        )
    )

if not temporal_analysis_path.exists():
    raise RuntimeError(
        "temporal_analysis_protocol.json was not saved."
    )

print(
    f"\nSaved:\n{temporal_analysis_path}"
)

print("\n" + "-" * 104)
print(
    "STAGE H: PASS — TEMPORAL-DEGRADATION "
    "ANALYSIS PRE-SPECIFIED"
)
print("=" * 104)

## Stage I — Source-Composition Confounding Control

Channel composition is treated as a diagnostic covariate for the temporal benchmark.

The primary research question remains temporal generalization:

Historical Dataset B → Frozen Dataset-B LoRA → Strictly future Dataset B windows

For historical train, historical validation, and F1–F6, record:

- source/channel counts;
- source/channel proportions;
- number of active sources;
- source composition relative to historical train.

Source-composition changes will be used to interpret temporal performance changes, not to redefine the benchmark.

A later source-controlled evaluation may be used as a sensitivity analysis if sample support is adequate.

In [ ]:
# ============================================================
# DAY 14 — CELL 15
# Stage I: source-composition diagnostics by split/window
# ============================================================

# ------------------------------------------------------------
# 1. Build canonical split dictionary
# ------------------------------------------------------------

split_frames = {
    "historical_train": train_df,
    "historical_validation": validation_df,
}

for window_id in [
    "F1", "F2", "F3",
    "F4", "F5", "F6"
]:
    split_frames[window_id] = (
        future_df.loc[
            future_df["future_window"] == window_id
        ]
        .copy()
    )

# ------------------------------------------------------------
# 2. Establish complete channel universe
# ------------------------------------------------------------

ALL_CHANNELS = sorted(
    benchmark_df[
        "channel_username"
    ]
    .dropna()
    .astype(str)
    .unique()
)

# ------------------------------------------------------------
# 3. Build long-format source composition table
# ------------------------------------------------------------

rows = []

for split_order, (split_name, df) in enumerate(
    split_frames.items(),
    start=1
):

    channel_counts = (
        df["channel_username"]
        .astype(str)
        .value_counts()
        .reindex(
            ALL_CHANNELS,
            fill_value=0
        )
    )

    for channel in ALL_CHANNELS:

        count = int(
            channel_counts.loc[channel]
        )

        proportion = (
            count / len(df)
            if len(df) > 0
            else np.nan
        )

        rows.append({
            "split_order":
                split_order,

            "split":
                split_name,

            "channel_username":
                channel,

            "channel_count":
                count,

            "channel_proportion":
                float(proportion),

            "split_n":
                int(len(df)),

            "active_in_split":
                bool(count > 0)
        })

source_composition_by_split = pd.DataFrame(
    rows
)

# ------------------------------------------------------------
# 4. Historical-train reference distribution
# ------------------------------------------------------------

train_reference = (
    source_composition_by_split.loc[
        source_composition_by_split[
            "split"
        ] == "historical_train",
        [
            "channel_username",
            "channel_proportion"
        ]
    ]
    .set_index("channel_username")[
        "channel_proportion"
    ]
)

# ------------------------------------------------------------
# 5. Compute source-composition TVD vs historical train
# ------------------------------------------------------------

source_tvd_rows = []

for split_name, df in split_frames.items():

    split_dist = (
        df["channel_username"]
        .astype(str)
        .value_counts()
        .reindex(
            ALL_CHANNELS,
            fill_value=0
        )
        .astype(float)
    )

    if split_dist.sum() > 0:
        split_dist = (
            split_dist /
            split_dist.sum()
        )

    tvd_vs_train = (
        0.5
        * np.abs(
            split_dist
            - train_reference
        ).sum()
    )

    train_active_sources = set(
        train_df[
            "channel_username"
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    split_active_sources = set(
        df[
            "channel_username"
        ]
        .dropna()
        .astype(str)
        .unique()
    )

    union = (
        train_active_sources
        | split_active_sources
    )

    intersection = (
        train_active_sources
        & split_active_sources
    )

    source_jaccard = (
        len(intersection) / len(union)
        if union else np.nan
    )

    source_tvd_rows.append({
        "split":
            split_name,

        "split_n":
            int(len(df)),

        "n_active_sources":
            int(len(split_active_sources)),

        "shared_sources_with_train":
            int(len(intersection)),

        "source_overlap_jaccard_vs_train":
            float(source_jaccard),

        "source_composition_tvd_vs_train":
            float(tvd_vs_train)
    })

source_shift_summary = pd.DataFrame(
    source_tvd_rows
)

# ------------------------------------------------------------
# 6. Validate long-format composition table
# ------------------------------------------------------------

proportion_sums = (
    source_composition_by_split
    .groupby("split")[
        "channel_proportion"
    ]
    .sum()
)

count_sums = (
    source_composition_by_split
    .groupby("split")[
        "channel_count"
    ]
    .sum()
)

expected_split_sizes = {
    name: len(df)
    for name, df
    in split_frames.items()
}

checks = {
    "eight_splits":
        len(split_frames) == 8,

    "all_eight_channels_represented":
        len(ALL_CHANNELS) == 8,

    "one_row_per_split_channel":
        len(source_composition_by_split)
        == len(split_frames) * len(ALL_CHANNELS),

    "proportions_sum_to_one":
        np.allclose(
            proportion_sums.values,
            1.0
        ),

    "counts_match_split_sizes":
        all(
            int(count_sums.loc[name])
            == expected_n
            for name, expected_n
            in expected_split_sizes.items()
        ),

    "train_tvd_zero":
        np.isclose(
            source_shift_summary.loc[
                source_shift_summary[
                    "split"
                ] == "historical_train",
                "source_composition_tvd_vs_train"
            ].iloc[0],
            0.0
        ),

    "future_windows_present":
        set(
            source_shift_summary[
                "split"
            ]
        )
        == {
            "historical_train",
            "historical_validation",
            "F1", "F2", "F3",
            "F4", "F5", "F6"
        }
}

# ------------------------------------------------------------
# 7. Save required artifact
# ------------------------------------------------------------

source_composition_path = (
    DAY14_DIR /
    "source_composition_by_split.csv"
)

source_composition_by_split.to_csv(
    source_composition_path,
    index=False
)

# Optional compact summary is useful downstream,
# but source_composition_by_split.csv is the required artifact.
source_shift_summary_path = (
    DAY14_DIR /
    "source_shift_summary.csv"
)

source_shift_summary.to_csv(
    source_shift_summary_path,
    index=False
)

# ------------------------------------------------------------
# 8. Report diagnostic summary
# ------------------------------------------------------------

print("=" * 116)
print("DAY 14 — STAGE I: SOURCE-COMPOSITION DIAGNOSTICS")
print("=" * 116)

print("\nSource universe:")
print(f"Channels represented : {len(ALL_CHANNELS)}")

print("\nComposition shift relative to historical train:\n")

print(
    source_shift_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nValidation checks:")

for name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status:<6} — {name}")

if not all(checks.values()):
    failed = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nStage I source-composition validation failed:\n"
        + "\n".join(
            f"- {name}"
            for name in failed
        )
    )

if not source_composition_path.exists():
    raise RuntimeError(
        "source_composition_by_split.csv was not saved."
    )

print(
    f"\nRequired artifact saved:\n"
    f"{source_composition_path}"
)

print(
    f"\nDiagnostic summary saved:\n"
    f"{source_shift_summary_path}"
)

print("\n" + "-" * 116)
print(
    "STAGE I: PASS — SOURCE COMPOSITION "
    "RECORDED FOR ALL SPLITS / WINDOWS"
)
print("=" * 116)

## Stage J — Final Boundary Leakage Guard

The Day 13 near-duplicate diagnostic is rerun using the exact same text normalization, TF-IDF representation, nearest-neighbor procedure, similarity threshold, and temporal threshold.

A suspicious pair is defined as:

- nearest-neighbor cosine similarity ≥ 0.95;
- absolute timestamp difference ≤ 24 hours;
- the two observations lie on opposite sides of a frozen chronological boundary.

Two frozen boundaries are checked:

1. **Historical Train → Historical Validation:** 2025-11-01 UTC
2. **Historical Development → Strictly Future Evaluation:** 2026-01-01 UTC

The 2026-01-01 boundary is the primary benchmark leakage guard.

High textual similarity alone is not treated as duplication because financial news may contain legitimate repeated or templated language across different events.

No broad similarity-based deletion is performed.

In [ ]:
# ============================================================
# DAY 14 — CELL 16
# Stage J: final exact-boundary duplicate / leakage audit
# ============================================================

import re
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors


# ------------------------------------------------------------
# 1. Reconstruct the frozen dense benchmark population
# ------------------------------------------------------------

leak_df = (
    benchmark_df
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

leak_df["_timestamp"] = pd.to_datetime(
    leak_df["date"],
    errors="coerce",
    utc=True
)

if leak_df["_timestamp"].isna().any():
    raise ValueError(
        "Invalid timestamps detected in frozen benchmark population."
    )


# ------------------------------------------------------------
# 2. EXACT Day 13 text normalization
# ------------------------------------------------------------

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


leak_df["_normalized_text"] = (
    leak_df["text"]
    .map(normalize_text)
)

exact_normalized_duplicate_rows = int(
    leak_df.duplicated(
        subset=["_normalized_text"],
        keep=False
    ).sum()
)


# ------------------------------------------------------------
# 3. EXACT Day 13 TF-IDF representation
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

X = vectorizer.fit_transform(
    leak_df["_normalized_text"]
)


# ------------------------------------------------------------
# 4. EXACT Day 13 nearest-neighbor procedure
# ------------------------------------------------------------

nn = NearestNeighbors(
    n_neighbors=2,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nn.fit(X)

distances, indices = nn.kneighbors(X)

nearest_similarity = (
    1.0 - distances[:, 1]
)

nearest_index = (
    indices[:, 1]
)

leak_df["_nearest_index"] = (
    nearest_index
)

leak_df["_nearest_similarity"] = (
    nearest_similarity
)

leak_df["_nearest_timestamp"] = (
    leak_df.iloc[
        nearest_index
    ]["_timestamp"]
    .to_numpy()
)

leak_df["_nearest_channel"] = (
    leak_df.iloc[
        nearest_index
    ]["channel_username"]
    .to_numpy()
)

leak_df["_time_difference_hours"] = (
    (
        leak_df["_timestamp"]
        - leak_df["_nearest_timestamp"]
    )
    .abs()
    .dt.total_seconds()
    / 3600
)


# ------------------------------------------------------------
# 5. Freeze Day 13 suspicious-pair definition
# ------------------------------------------------------------

SIM_THRESHOLD = 0.95
TIME_WINDOW_HOURS = 24.0


# ------------------------------------------------------------
# 6. Overall diagnostic prevalence
# ------------------------------------------------------------

short_horizon_high_sim = (
    (
        leak_df["_nearest_similarity"]
        >= SIM_THRESHOLD
    )
    &
    (
        leak_df["_time_difference_hours"]
        <= TIME_WINDOW_HOURS
    )
)

short_horizon_n = int(
    short_horizon_high_sim.sum()
)

short_horizon_share = float(
    short_horizon_n
    / len(leak_df)
)


# ------------------------------------------------------------
# 7. Exact frozen boundaries
# ------------------------------------------------------------

FINAL_BOUNDARIES = {
    "train_to_validation":
        VALIDATION_START,

    "development_to_future":
        SELECTED_CUTOFF,
}


def audit_boundary(df, boundary_name, boundary):
    """
    Apply the exact Day 13 cross-boundary suspicious-pair definition.
    """

    current_before = (
        df["_timestamp"]
        < boundary
    )

    neighbor_before = (
        df["_nearest_timestamp"]
        < boundary
    )

    crosses_boundary = (
        current_before
        != neighbor_before
    )

    suspicious = (
        crosses_boundary
        &
        (
            df["_nearest_similarity"]
            >= SIM_THRESHOLD
        )
        &
        (
            df["_time_difference_hours"]
            <= TIME_WINDOW_HOURS
        )
    )

    suspicious_rows = int(
        suspicious.sum()
    )

    suspicious_indices = np.where(
        suspicious.to_numpy()
    )[0]

    pair_ids = set()

    for i in suspicious_indices:

        j = int(
            df.iloc[i][
                "_nearest_index"
            ]
        )

        pair_ids.add(
            tuple(
                sorted(
                    (
                        int(i),
                        j
                    )
                )
            )
        )

    suspicious_pairs = len(
        pair_ids
    )

    before_n = int(
        (
            df["_timestamp"]
            < boundary
        ).sum()
    )

    after_n = int(
        (
            df["_timestamp"]
            >= boundary
        ).sum()
    )

    return {
        "boundary_name":
            boundary_name,

        "boundary_utc":
            boundary.isoformat(),

        "before_n":
            before_n,

        "after_n":
            after_n,

        "cross_boundary_suspicious_rows":
            suspicious_rows,

        "cross_boundary_unique_pairs":
            suspicious_pairs,

        "similarity_threshold":
            SIM_THRESHOLD,

        "time_window_hours":
            TIME_WINDOW_HOURS,

        "gate":
            (
                "PASS"
                if suspicious_pairs == 0
                else "FAIL"
            )
    }


boundary_results = [
    audit_boundary(
        leak_df,
        name,
        boundary
    )
    for name, boundary
    in FINAL_BOUNDARIES.items()
]

boundary_audit_df = pd.DataFrame(
    boundary_results
)


# ------------------------------------------------------------
# 8. Main final-boundary result
# ------------------------------------------------------------

primary_result = next(
    result
    for result in boundary_results
    if result["boundary_name"]
    == "development_to_future"
)

train_validation_result = next(
    result
    for result in boundary_results
    if result["boundary_name"]
    == "train_to_validation"
)


# ------------------------------------------------------------
# 9. Formal final audit artifact
# ------------------------------------------------------------

final_boundary_leakage_audit = {
    "day":
        14,

    "artifact":
        "final_boundary_leakage_audit",

    "dataset_population": {
        "population":
            "frozen_dense_benchmark",

        "n":
            int(len(leak_df)),

        "start":
            leak_df[
                "_timestamp"
            ].min().isoformat(),

        "end":
            leak_df[
                "_timestamp"
            ].max().isoformat()
    },

    "implementation": {
        "reproduces_day13_definition":
            True,

        "text_normalization": [
            "lowercase",
            "remove URLs",
            "collapse whitespace",
            "strip whitespace"
        ],

        "tfidf": {
            "analyzer":
                "char_wb",

            "ngram_range":
                [3, 5],

            "min_df":
                2,

            "max_features":
                100000,

            "sublinear_tf":
                True
        },

        "nearest_neighbor": {
            "n_neighbors":
                2,

            "metric":
                "cosine",

            "algorithm":
                "brute",

            "neighbor_used":
                "nearest non-self neighbor"
        }
    },

    "exact_duplicate_audit": {
        "exact_normalized_text_duplicate_rows":
            exact_normalized_duplicate_rows,

        "gate":
            (
                "PASS"
                if exact_normalized_duplicate_rows == 0
                else "REVIEW"
            )
    },

    "near_duplicate_definition": {
        "similarity_threshold":
            SIM_THRESHOLD,

        "maximum_time_difference_hours":
            TIME_WINDOW_HOURS,

        "interpretation":
            (
                "Diagnostic flag for likely repost or lightly "
                "reformulated same-event observations; not a "
                "general duplicate-removal rule."
            )
    },

    "overall_near_duplicate_diagnostic": {
        "nearest_similarity_ge_0_90":
            int(
                (
                    leak_df[
                        "_nearest_similarity"
                    ] >= 0.90
                ).sum()
            ),

        "nearest_similarity_ge_0_95":
            int(
                (
                    leak_df[
                        "_nearest_similarity"
                    ] >= 0.95
                ).sum()
            ),

        "nearest_similarity_ge_0_98":
            int(
                (
                    leak_df[
                        "_nearest_similarity"
                    ] >= 0.98
                ).sum()
            ),

        "short_horizon_high_similarity_rows":
            short_horizon_n,

        "short_horizon_high_similarity_share":
            short_horizon_share
    },

    "frozen_boundary_tests":
        boundary_results,

    "primary_benchmark_boundary": {
        "boundary_name":
            "development_to_future",

        "boundary_utc":
            SELECTED_CUTOFF.isoformat(),

        "cross_boundary_unique_pairs":
            int(
                primary_result[
                    "cross_boundary_unique_pairs"
                ]
            ),

        "gate":
            primary_result["gate"]
    },

    "train_validation_boundary": {
        "boundary_utc":
            VALIDATION_START.isoformat(),

        "cross_boundary_unique_pairs":
            int(
                train_validation_result[
                    "cross_boundary_unique_pairs"
                ]
            ),

        "gate":
            train_validation_result["gate"]
    },

    "resolution_rule": (
        "If any suspicious cross-boundary pair exists, inspect and "
        "document that pair before training. Do not perform broad "
        "similarity-based deletion."
    ),

    "day14_final_boundary_leakage_gate": (
        "PASS"
        if (
            primary_result["gate"] == "PASS"
            and
            train_validation_result["gate"] == "PASS"
        )
        else "FAIL"
    )
}


# ------------------------------------------------------------
# 10. Save artifact
# ------------------------------------------------------------

final_leakage_path = (
    DAY14_DIR
    / "final_boundary_leakage_audit.json"
)

with open(
    final_leakage_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_boundary_leakage_audit,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 11. Final validation checks
# ------------------------------------------------------------

checks = {
    "benchmark_n_39400":
        len(leak_df) == 39_400,

    "exact_duplicates_match_day13":
        exact_normalized_duplicate_rows
        == 0,

    "short_horizon_rows_match_day13":
        short_horizon_n
        == 130,

    "train_validation_boundary_pass":
        train_validation_result[
            "gate"
        ] == "PASS",

    "development_future_boundary_pass":
        primary_result[
            "gate"
        ] == "PASS",

    "primary_cross_boundary_pairs_zero":
        primary_result[
            "cross_boundary_unique_pairs"
        ] == 0,

    "artifact_saved":
        final_leakage_path.exists()
}


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print("=" * 108)
print("DAY 14 — STAGE J: FINAL BOUNDARY LEAKAGE GUARD")
print("=" * 108)

print(
    f"\nDense benchmark N                 : "
    f"{len(leak_df):,}"
)

print(
    f"Exact normalized duplicates      : "
    f"{exact_normalized_duplicate_rows:,}"
)

print(
    f"Short-horizon high-similarity    : "
    f"{short_horizon_n:,} "
    f"({short_horizon_share:.4%})"
)

print("\nFrozen boundary tests:\n")

print(
    boundary_audit_df[
        [
            "boundary_name",
            "boundary_utc",
            "before_n",
            "after_n",
            "cross_boundary_suspicious_rows",
            "cross_boundary_unique_pairs",
            "gate"
        ]
    ].to_string(
        index=False
    )
)

print("\nValidation checks:")

for name, passed in checks.items():
    status = (
        "PASS"
        if passed
        else "FAIL"
    )

    print(
        f"{status:<6} — {name}"
    )

if not all(
    checks.values()
):

    failed = [
        name
        for name, passed
        in checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nStage J final-boundary leakage guard failed:\n"
        + "\n".join(
            f"- {name}"
            for name in failed
        )
    )

print(
    f"\nSaved:\n"
    f"{final_leakage_path}"
)

print("\n" + "-" * 108)

print(
    "STAGE J: PASS — FINAL CHRONOLOGICAL "
    "BOUNDARIES PASS LEAKAGE GUARD"
)

print("=" * 108)

## Day 14 Final Checkpoint

All benchmark-design decisions have now been frozen before any new Dataset-B LoRA training.

The final checkpoint verifies:

- benchmark population;
- historical cutoff;
- chronological train / validation split;
- future windows F1–F6;
- LoRA training protocol;
- evaluation protocol;
- temporal-degradation analysis;
- source-composition diagnostics;
- final-boundary leakage guard;
- required Day 14 artifacts.

If all checks pass, Day 14 status is:

**PASS — CHRONOLOGICAL PROTOCOL FROZEN**

Only after this checkpoint should Day 15 begin Dataset-B historical LoRA training.

In [ ]:
# ============================================================
# DAY 14 — CELL 17
# Final acceptance checkpoint and protocol summary
# ============================================================

import json
from pathlib import Path

# ------------------------------------------------------------
# 1. Required Day 14 artifacts
# ------------------------------------------------------------

required_artifacts = {
    "benchmark_population_manifest.json":
        DAY14_DIR / "benchmark_population_manifest.json",

    "candidate_cutoff_feasibility.csv":
        DAY14_DIR / "candidate_cutoff_feasibility.csv",

    "chronological_split_spec.json":
        DAY14_DIR / "chronological_split_spec.json",

    "future_window_manifest.csv":
        DAY14_DIR / "future_window_manifest.csv",

    "source_composition_by_split.csv":
        DAY14_DIR / "source_composition_by_split.csv",

    "lora_training_protocol.json":
        DAY14_DIR / "lora_training_protocol.json",

    "evaluation_protocol.json":
        DAY14_DIR / "evaluation_protocol.json",

    "final_boundary_leakage_audit.json":
        DAY14_DIR / "final_boundary_leakage_audit.json",
}

# Additional formal protocol artifact created in Stage H
additional_artifacts = {
    "temporal_analysis_protocol.json":
        DAY14_DIR / "temporal_analysis_protocol.json",

    "source_shift_summary.csv":
        DAY14_DIR / "source_shift_summary.csv",
}


# ------------------------------------------------------------
# 2. Artifact existence checks
# ------------------------------------------------------------

artifact_checks = {
    name: path.exists()
    for name, path
    in required_artifacts.items()
}

additional_artifact_checks = {
    name: path.exists()
    for name, path
    in additional_artifacts.items()
}


# ------------------------------------------------------------
# 3. Acceptance criteria
# ------------------------------------------------------------

acceptance_checks = {
    # Population
    "dense_benchmark_population_39400":
        len(benchmark_df) == 39_400,

    # Historical cutoff
    "historical_cutoff_frozen_2026_01_01":
        SELECTED_CUTOFF
        == pd.Timestamp(
            "2026-01-01",
            tz="UTC"
        ),

    # Train / validation chronology
    "historical_train_precedes_validation":
        train_df["date"].max()
        < validation_df["date"].min(),

    "historical_validation_precedes_future":
        validation_df["date"].max()
        < future_df["date"].min(),

    # Frozen split sizes
    "historical_train_n_13539":
        len(train_df) == 13_539,

    "historical_validation_n_5859":
        len(validation_df) == 5_859,

    "future_total_n_20002":
        len(future_df) == 20_002,

    # Future windows
    "six_future_windows":
        future_window_manifest[
            "future_window"
        ].tolist()
        == [
            "F1", "F2", "F3",
            "F4", "F5", "F6"
        ],

    "future_windows_preserve_all_rows":
        int(
            future_window_manifest[
                "n"
            ].sum()
        ) == 20_002,

    "every_future_window_all_classes":
        (
            future_window_manifest[
                "min_class_n"
            ] > 0
        ).all(),

    "f6_explicitly_partial":
        bool(
            future_window_manifest.loc[
                future_window_manifest[
                    "future_window"
                ] == "F6",
                "is_partial_month"
            ].iloc[0]
        ),

    # Split-selection integrity
    "split_selection_no_model_performance":
        True,

    # LoRA protocol
    "lora_protocol_frozen":
        lora_training_protocol[
            "freeze_status"
        ] == "LOCKED_BEFORE_DAY15",

    "future_excluded_from_model_selection":
        set(
            lora_training_protocol[
                "data_protocol"
            ]["future_windows_prohibited"]
        )
        == {
            "F1", "F2", "F3",
            "F4", "F5", "F6"
        },

    # Evaluation protocol
    "primary_metric_macro_f1":
        evaluation_protocol[
            "metrics"
        ]["primary"]["name"]
        == "macro_f1",

    "secondary_metrics_frozen":
        {
            x["name"]
            for x in evaluation_protocol[
                "metrics"
            ]["secondary"]
        }
        == {
            "accuracy",
            "weighted_f1"
        },

    # Temporal analysis
    "temporal_analysis_frozen":
        temporal_analysis_protocol[
            "freeze_status"
        ]
        == "LOCKED_BEFORE_FUTURE_INFERENCE",

    "non_monotonic_temporal_analysis":
        temporal_analysis_protocol[
            "analysis_scope"
        ]["monotonic_degradation_required"]
        is False,

    # Source diagnostics
    "source_composition_recorded_for_all_splits":
        set(
            source_composition_by_split[
                "split"
            ].unique()
        )
        == {
            "historical_train",
            "historical_validation",
            "F1", "F2", "F3",
            "F4", "F5", "F6"
        },

    # Leakage guard
    "train_validation_leakage_guard_pass":
        train_validation_result[
            "gate"
        ] == "PASS",

    "development_future_leakage_guard_pass":
        primary_result[
            "gate"
        ] == "PASS",

    "zero_primary_cross_boundary_pairs":
        primary_result[
            "cross_boundary_unique_pairs"
        ] == 0,

    # Required artifacts
    "all_required_artifacts_exist":
        all(
            artifact_checks.values()
        )
}


# ------------------------------------------------------------
# 4. Overall status
# ------------------------------------------------------------

all_acceptance_pass = all(
    acceptance_checks.values()
)

day14_status = (
    "PASS — CHRONOLOGICAL PROTOCOL FROZEN"
    if all_acceptance_pass
    else "FAIL — PROTOCOL INCOMPLETE"
)


# ------------------------------------------------------------
# 5. Build final Day 14 summary
# ------------------------------------------------------------

day14_protocol_summary = {
    "day":
        14,

    "project":
        "Financial Language Models under Temporal Distribution Shift",

    "objective":
        (
            "Freeze a within-Dataset-B chronological benchmark "
            "before any new Dataset-B LoRA training."
        ),

    "status":
        day14_status,

    "benchmark_population": {
        "n":
            int(len(benchmark_df)),

        "start_utc":
            benchmark_df[
                "date"
            ].min().isoformat(),

        "end_utc":
            benchmark_df[
                "date"
            ].max().isoformat(),

        "excluded_sparse_early_n":
            int(len(excluded_df))
    },

    "chronological_design": {
        "historical_cutoff_utc":
            SELECTED_CUTOFF.isoformat(),

        "historical_train": {
            "n":
                int(len(train_df)),

            "start_utc":
                train_df[
                    "date"
                ].min().isoformat(),

            "end_utc":
                train_df[
                    "date"
                ].max().isoformat()
        },

        "historical_validation": {
            "n":
                int(len(validation_df)),

            "start_utc":
                validation_df[
                    "date"
                ].min().isoformat(),

            "end_utc":
                validation_df[
                    "date"
                ].max().isoformat()
        },

        "strictly_future": {
            "n":
                int(len(future_df)),

            "windows":
                [
                    "F1", "F2", "F3",
                    "F4", "F5", "F6"
                ],

            "F6_partial":
                True
        }
    },

    "model_protocol": {
        "base_model":
            lora_training_protocol[
                "base_model"
            ]["model_name"],

        "max_steps":
            lora_training_protocol[
                "training"
            ]["max_steps"],

        "checkpoint_selection":
            (
                "historical-validation eval_loss only"
            ),

        "future_data_used_for_model_selection":
            False
    },

    "evaluation_protocol": {
        "primary_metric":
            "macro_f1",

        "secondary_metrics": [
            "accuracy",
            "weighted_f1"
        ],

        "class_specific_recall":
            True,

        "confusion_transitions":
            True,

        "prediction_distribution":
            True
    },

    "temporal_analysis": {
        "each_future_window_independent":
            True,

        "change_vs_validation":
            True,

        "change_vs_F1":
            True,

        "continuous_time_predictor":
            "days_from_development_end_to_midpoint",

        "monotonic_degradation_required":
            False
    },

    "source_diagnostics": {
        "channel_composition_tracked":
            True,

        "source_controlled_analysis_role":
            "later sensitivity analysis only"
    },

    "leakage_guard": {
        "similarity_threshold":
            SIM_THRESHOLD,

        "time_window_hours":
            TIME_WINDOW_HOURS,

        "train_validation_cross_boundary_pairs":
            int(
                train_validation_result[
                    "cross_boundary_unique_pairs"
                ]
            ),

        "development_future_cross_boundary_pairs":
            int(
                primary_result[
                    "cross_boundary_unique_pairs"
                ]
            ),

        "gate":
            final_boundary_leakage_audit[
                "day14_final_boundary_leakage_gate"
            ]
    },

    "required_artifacts": {
        name: bool(exists)
        for name, exists
        in artifact_checks.items()
    },

    "additional_artifacts": {
        name: bool(exists)
        for name, exists
        in additional_artifact_checks.items()
    },

    "acceptance_checks": {
        name: bool(passed)
        for name, passed
        in acceptance_checks.items()
    },

    "approved_next_stage": (
        "Day 15 — train the frozen LoRA configuration "
        "using Dataset B historical train only, with historical "
        "validation used only for checkpoint selection."
        if all_acceptance_pass
        else None
    )
}


# ------------------------------------------------------------
# 6. Save final summary artifact
# ------------------------------------------------------------

day14_summary_path = (
    DAY14_DIR /
    "day14_protocol_summary.json"
)

with open(
    day14_summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        day14_protocol_summary,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 7. Final report
# ------------------------------------------------------------

print("=" * 108)
print("DAY 14 — FINAL ARTIFACT CHECKPOINT")
print("=" * 108)

print("\nRequired artifacts:")

for name, exists in artifact_checks.items():

    print(
        f"{'PASS' if exists else 'FAIL':<6} — "
        f"{name}"
    )

print("\nAdditional retained artifacts:")

for name, exists in (
    additional_artifact_checks.items()
):

    print(
        f"{'PASS' if exists else 'FAIL':<6} — "
        f"{name}"
    )

print("\nAcceptance criteria:")

for name, passed in (
    acceptance_checks.items()
):

    print(
        f"{'PASS' if passed else 'FAIL':<6} — "
        f"{name}"
    )

print("\nFrozen benchmark:")
print(
    f"Historical Train      : "
    f"{len(train_df):,}"
)
print(
    f"Historical Validation : "
    f"{len(validation_df):,}"
)
print(
    f"Strictly Future       : "
    f"{len(future_df):,}"
)
print(
    "Future windows        : "
    "F1, F2, F3, F4, F5, F6"
)

print("\nFinal leakage guard:")
print(
    f"Train -> Validation pairs : "
    f"{train_validation_result['cross_boundary_unique_pairs']}"
)
print(
    f"Development -> Future pairs: "
    f"{primary_result['cross_boundary_unique_pairs']}"
)

print(
    f"\nSaved:\n"
    f"{day14_summary_path}"
)

print("\n" + "=" * 108)
print(f"DAY 14 STATUS: {day14_status}")
print("=" * 108)

if not all_acceptance_pass:

    failed = [
        name
        for name, passed
        in acceptance_checks.items()
        if not passed
    ]

    raise AssertionError(
        "\nDay 14 final acceptance failed:\n"
        + "\n".join(
            f"- {name}"
            for name in failed
        )
    )